# Hybrid Work and Job–Home Networks in Spain
## Systematic mapping workflow, public release (v4.2)

This notebook is the cartographic companion to:

`01_Hybrid_Work_Job_Home_Networks_Formal_Analysis_v4_4_public.ipynb`

Set the `HYBRID_WORK_PROJECT_ROOT` environment variable to the project directory before running the notebook. If the variable is not set, the notebook uses the current working directory. The shared workflow expects the following layout:

```text
hybrid_work_project/
├── basemaps/
├── 03_reference/
├── 05_analysis/
│   └── 13_hybrid_network_article/
└── 07_collected_outputs/
```

The mapping workflow separates descriptive coverage from regression-sample coverage.

1. **All six intensity-layer networks**
   - one main figure arranged in three rows and two columns;
   - six separate appendix maps;
   - one identical saturated network color for every layer;
   - identical edge selection, line-width scaling, and flow legend.

2. **Maximum-available district maps**
   - all-flow hybrid-work intensity;
   - period partner coverage, partner diversity, and mean external distance;
   - the six outcome panels are arranged in three rows and two columns;
   - structural zeros are retained for coverage and diversity;
   - true missing values are hatched only when the variable is genuinely unavailable.

3. **Settlement context and conditional relationships**
   - one national settlement-urbanity map;
   - residential and employment onsite-position maps;
   - model-sample marginal-effect maps arranged in three rows and two columns;
   - explicit outcome-name reconciliation with Notebook 01 prevents valid partner-diversity estimates from being misclassified as missing.

4. **Bivariate appendix**
   - 3 × 3 bivariate legends and the color structure used in the earlier bivariate-appendix module;
   - hybrid-work intensity combined with the three network outcomes;
   - hybrid-work intensity combined with urbanity and onsite network position.

The Canary Islands inset and fixed-decimal legends are retained.


## 1. Dependencies

In [ ]:
# Run once if the environment does not already contain these packages.
%pip install -q pandas pyarrow numpy matplotlib geopandas pyogrio shapely mapclassify

## 2. Configuration

In [ ]:
import os
from pathlib import Path, PureWindowsPath

# ---------------------------------------------------------------------
# Project locations
# ---------------------------------------------------------------------
# Set HYBRID_WORK_PROJECT_ROOT to the shared project directory. When it is
# absent, the current working directory is used as the project root.
PROJECT_ROOT = Path(
    os.environ.get("HYBRID_WORK_PROJECT_ROOT", Path.cwd())
).expanduser().resolve()
ARTICLE_ROOT = (
    PROJECT_ROOT
    / "05_analysis"
    / "13_hybrid_network_article"
)


def portable_path_label(value):
    """Return project-relative provenance without exposing local paths."""
    if value is None:
        return None

    text = str(value)
    candidate = Path(text)
    try:
        return str(candidate.relative_to(PROJECT_ROOT))
    except ValueError:
        if "\\" in text or (len(text) > 1 and text[1] == ":"):
            return PureWindowsPath(text).name
        return candidate.name

# Notebook-01 source version required by this mapping run.
FORMAL_RUN_TAG = "v4_4_20260724"
FORMAL_STAGE_DIR = (
    ARTICLE_ROOT
    / f"01_intensity_gradient_analysis_{FORMAL_RUN_TAG}"
)

TABLE_MAIN_DIR = FORMAL_STAGE_DIR / "tables_main"
TABLE_APPENDIX_DIR = FORMAL_STAGE_DIR / "tables_appendix"
MAP_INPUT_DIR = FORMAL_STAGE_DIR / "map_inputs_for_notebook_02"

BASEMAP_DIR = PROJECT_ROOT / "basemaps"
REFERENCE_DIR = PROJECT_ROOT / "03_reference"

# Independent version and date for all Notebook-02 outputs.
MAPPING_RUN_VERSION = "v4_2"
MAPPING_RUN_DATE = "20260724"
MAPPING_RUN_TAG = f"{MAPPING_RUN_VERSION}_{MAPPING_RUN_DATE}"
MAPPING_STAGE_DIR = (
    ARTICLE_ROOT
    / f"02_intensity_gradient_mapping_{MAPPING_RUN_TAG}"
)
FIGURE_MAIN_DIR = MAPPING_STAGE_DIR / "figures_main"
FIGURE_APPENDIX_DIR = MAPPING_STAGE_DIR / "figures_appendix"
DATA_EXPORT_DIR = MAPPING_STAGE_DIR / "map_data"
LOG_DIR = MAPPING_STAGE_DIR / "logs"

for folder in [
    MAPPING_STAGE_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_APPENDIX_DIR,
    DATA_EXPORT_DIR,
    LOG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Required Notebook-01 outputs
# ---------------------------------------------------------------------
PERIOD_LAYER_EDGE_FILE = MAP_INPUT_DIR / "period_layer_network_edges_map_ready.parquet"
DISTRICT_PERIOD_FILE = MAP_INPUT_DIR / "district_period_role_all_available_metrics_map_ready.parquet"
DISTRICT_MONTH_FILE = TABLE_MAIN_DIR / "district_month_role_intensity_metrics.parquet"
MARGINAL_EFFECT_FILE = MAP_INPUT_DIR / "district_context_marginal_intensity_effects_map_ready.parquet"
CENTROID_LOOKUP_FILE = TABLE_APPENDIX_DIR / "district_centroid_lookup_epsg3035.parquet"
PAIRWISE_SIMILARITY_FILE = TABLE_MAIN_DIR / "period_pairwise_layer_similarity.csv"

FORMAL_LOG_DIR = FORMAL_STAGE_DIR / "logs"
FORMAL_MANIFEST_FILE = FORMAL_LOG_DIR / "analysis_manifest.json"
GHS_PROFILE_AUDIT_FILE = FORMAL_LOG_DIR / "ghs_profile_audit.csv"
PERIOD_GHS_AUDIT_FILE = FORMAL_LOG_DIR / "period_district_ghs_merge_audit.csv"

RECURRENCE_BAND_ORDER = ["1-2", "3-4", "5-7", "8-10", "11-13", "14"]
RECURRENCE_MIDPOINT = {
    "1-2": 1.5, "3-4": 3.5, "5-7": 6.0,
    "8-10": 9.0, "11-13": 12.0, "14": 14.0,
}
RECURRENCE_INTENSITY = {
    band: (14.0 - midpoint) / (14.0 - 1.5)
    for band, midpoint in RECURRENCE_MIDPOINT.items()
}
MAIN_NETWORK_MAP_BANDS = RECURRENCE_BAND_ORDER.copy()
EXPORT_INDIVIDUAL_LAYER_MAPS = True

BAND_DISPLAY_LABELS = {
    "1-2": "1–2",
    "3-4": "3–4",
    "5-7": "5–7",
    "8-10": "8–10",
    "11-13": "11–13",
    "14": "14",
}

LAYER_FILE_LABELS = {
    "1-2": "01_1_2_days",
    "3-4": "02_3_4_days",
    "5-7": "03_5_7_days",
    "8-10": "04_8_10_days",
    "11-13": "05_11_13_days",
    "14": "06_14_days",
}

# ---------------------------------------------------------------------
# Spatial files
# ---------------------------------------------------------------------
DISTRICT_POLYGON_PREFERRED_NAMES = [
    "zonificacion_distritos.shp",
    "zonificacion_distritos_poligonos.shp",
    "zonificacion_distritos_polygon.shp",
]

# Province boundaries are derived from filtered Spanish district polygons.
DERIVE_PROVINCES_FROM_DISTRICT_IDS = True
PROVINCE_CODE_DIGITS = 2
MIN_PROVINCE_PREFIX_COVERAGE = 0.90

MAP_CRS = "EPSG:3035"
DISPLAY_CRS = MAP_CRS


SPAIN_DISTRICT_ID_PATTERN = r"^(?:0[1-9]|[1-4][0-9]|5[0-2])"
MIN_FORMAL_ID_POLYGON_COVERAGE = 0.999
MIN_NETWORK_MAP_VALUE_COVERAGE = 0.995
FAIL_ON_COVERAGE_ERROR = True
FILL_STRUCTURAL_NETWORK_ZEROS = True
SHOW_TRUE_MISSING_AS_HATCH = True
TRUE_MISSING_HATCH = "////"
TRUE_MISSING_EDGE = "#666666"
TRUE_MISSING_LABEL = "No observed value"

# ---------------------------------------------------------------------
# Figure style
# ---------------------------------------------------------------------
FIGURE_DPI = 400
FONT_FAMILY = "Times New Roman"
BASE_FIGSIZE = (8.2, 7.2)
PANEL_FIGSIZE = (15.2, 9.6)

DISTRICT_FACE = "#f2f2f2"
DISTRICT_EDGE = "#bdbdbd"
PROVINCE_EDGE = "#6f6f6f"
NO_DATA_FACE = "#e6e6e6"

HYBRID_COLOUR = "#9c2f2f"
ONSITE_COLOUR = "#255f85"
SHARED_COLOUR = "#5c5c5c"

ROLE_LABELS = {
    "residential": "Residential role",
    "employment": "Employment role",
}

LAYER_NETWORK_COLOUR = "#1f5a91"
LAYER_COLOURS = {
    band: LAYER_NETWORK_COLOUR
    for band in RECURRENCE_BAND_ORDER
}
HYBRID_LIGHT = "#d99a94"
ONSITE_LIGHT = "#93b8cf"

SEQUENTIAL_CMAP = "viridis"
HYBRID_CMAP = "Reds"
ONSITE_CMAP = "Blues"
DIVERGING_CMAP = "RdBu_r"
URBANITY_CMAP = "YlOrBr"
HIERARCHY_CMAP = "PuBu"


# ---------------------------------------------------------------------
# Canary Islands inset
# ---------------------------------------------------------------------
ENABLE_CANARY_INSET = True

CANARY_LONGITUDE_MAX = -10.0
CANARY_LATITUDE_MAX = 31.0

# Base position and size in parent-axis fractions.
CANARY_INSET_BASE_LEFT = 0.025
CANARY_INSET_BASE_BOTTOM = 0.035
CANARY_INSET_WIDTH = 0.285
CANARY_INSET_HEIGHT = 0.245

# Manual controls in kilometres.
# Positive X = right; negative X = left.
# Positive Y = up; negative Y = down.
CANARY_INSET_SHIFT_X_KM = 0.0
CANARY_INSET_SHIFT_Y_KM = -150.0

CANARY_INSET_LABEL = None
CANARY_INSET_SUBTITLE = None
CANARY_INSET_SHOW_SUBTITLE = False

MAIN_EXTENT_PADDING_SHARE = 0.025
CANARY_EXTENT_PADDING_SHARE = 0.080

INSET_BORDER_COLOUR = "#4f4f4f"
INSET_BORDER_WIDTH = 0.8
INSET_LABEL_FONTSIZE = 8
INSET_SCALE_BAR = True

CROSS_WINDOW_EDGE_POLICY = "omit"
MAIN_SCALE_BAR_LOCATION = (0.46, 0.048)


# ---------------------------------------------------------------------
# Flow-map controls
# ---------------------------------------------------------------------
# Default: select every positive inter-district edge.
#
# Available selection modes:
#   "all"                all positive inter-district edges
#   "top_n"              strongest N edges
#   "weight_quantile"    edges above a selected quantile
#   "cumulative_share"   strongest edges accounting for a target share
#   "coverage"           global top-N plus top-K by origin/destination
FLOW_FILTER_MODE = "all"

# Rendering is independent from edge selection.
#
# Available rendering modes:
#   "all_plus_backbone"  all edges faintly, strongest edges emphasised
#   "weighted_all"       all selected edges use weighted line widths
#   "uniform_all"        all selected edges use one uniform line width
FLOW_RENDER_MODE = "all_plus_backbone"

# Figure 1 can use normalised edge shares or original weights.
# "share" is the recommended default for horizontal comparison.
NATIONAL_NETWORK_MEASURE = "share"  # "share" or "raw_weight"

# Self-loops remain in the analytical data but have zero map length.
FLOW_MAP_EXCLUDE_SELF_LOOPS = True

# Parameters used only by non-"all" selection modes.
NATIONAL_TOP_N = 5000
PANEL_TOP_N = 4000
EDGE_WEIGHT_QUANTILE = 0.95
CUMULATIVE_EDGE_MASS = 0.90

FLOW_COVERAGE_GLOBAL_TOP_N = 2500
FLOW_COVERAGE_TOP_K_OUT = 2
FLOW_COVERAGE_TOP_K_IN = 2
FLOW_COVERAGE_MAX_EDGES = 6000

# Keep at zero to retain all positive inter-district edges.
FLOW_MIN_WEIGHT = 0.0

# All-edge contextual layer.
ALL_EDGE_WIDTH = 0.06
ALL_EDGE_ALPHA = 0.022

# Emphasised backbone layer.
FLOW_BACKBONE_QUANTILE = 0.985
BACKBONE_MIN_WIDTH = 0.10
BACKBONE_MAX_WIDTH = 1.50
BACKBONE_ALPHA = 0.30

# Alternative weighted-all rendering.
WEIGHTED_ALL_ALPHA = 0.10

# Width transformation and legend values.
FLOW_LINEWIDTH_SCALE = "log1p"  # log1p, sqrt or linear
FLOW_LOG_STRETCH = 100.0
FLOW_LINEWIDTH_CAP_QUANTILE = 0.995
FLOW_LEGEND_QUANTILES = (0.50, 0.90, 0.99)

FLOW_RASTERIZED = True

BACKGROUND_EDGEWIDTH = 0.08
PROVINCE_EDGEWIDTH = 0.45

# Difference maps use normalised edge-weight differences.
DIFFERENCE_TOP_N = 4000
BALANCED_TOLERANCE = 1e-12

# Edge-class map uses the same selection and width scale, but keeps its
# category colours.
EDGE_CLASS_RENDER_MODE = "all_plus_backbone"

# Optional nodes can be added later without changing edge selection.
SHOW_NETWORK_NODES = False
NODE_STRENGTH_QUANTILE = 0.95


# ---------------------------------------------------------------------
# Six-layer national-network figure layout
# ---------------------------------------------------------------------
ALL_LAYER_FIGSIZE = (14.6, 18.0)
ALL_LAYER_TOP = 0.945
ALL_LAYER_BOTTOM = 0.045
ALL_LAYER_RIGHT = 0.855
ALL_LAYER_WSPACE = 0.050
ALL_LAYER_HSPACE = 0.180
ALL_LAYER_LEGEND_ANCHOR = (0.985, 0.50)

# Shared layout for six-panel district figures. The visual arrangement is
# three rows by two columns: outcomes vary by row and roles vary by column.
SIX_PANEL_FIGSIZE = (14.6, 17.2)
SIX_PANEL_TOP = 0.940
SIX_PANEL_BOTTOM = 0.045
SIX_PANEL_RIGHT = 0.875
SIX_PANEL_WSPACE = 0.070
SIX_PANEL_HSPACE = 0.190

INDIVIDUAL_LAYER_FIGSIZE = (9.2, 7.7)
INDIVIDUAL_LAYER_RIGHT = 0.815
INDIVIDUAL_LAYER_LEGEND_ANCHOR = (0.985, 0.50)

FLOW_LEGEND_MAX_DECIMALS = 8
COLOURBAR_MAX_DECIMALS = 4

RESET_LAYER_MAP_OUTPUTS = True


# ---------------------------------------------------------------------
# Bivariate appendix
# ---------------------------------------------------------------------
RUN_BIVARIATE_APPENDIX = True
BIVARIATE_QUANTILES = (1 / 3, 2 / 3)
BIVARIATE_COLOURS = [
    "#e8e8e8",
    "#ace4e4",
    "#5ac8c8",
    "#dfb0d6",
    "#a5add3",
    "#5698b9",
    "#be64ac",
    "#8c62aa",
    "#3b4994",
]
BIVARIATE_LEGEND_POSITION = [0.66, 0.045, 0.28, 0.25]

# ---------------------------------------------------------------------
# Choropleth controls
# ---------------------------------------------------------------------
N_CLASSES = 7
SEQUENTIAL_CLASSIFICATION = "quantiles"  # quantiles, equal_interval, fisher_jenks
DIVERGING_LIMIT_QUANTILE = 0.98
CLIP_SEQUENTIAL_QUANTILES = (0.01, 0.99)
MISSING_LABEL = "No data"


# Directly comparable Hybrid/Onsite choropleths use one common
# sequential colour map and one common colour bar.
PAIRED_SEQUENTIAL_CMAP = "viridis"

# Explicit map-figure layouts. These leave room for external Canary
# insets and figure-level legends.
FIGURE_TOP = 0.88
FIGURE_BOTTOM = 0.08
FIGURE_LEFT = 0.025
FIGURE_RIGHT_STANDARD = 0.975
FIGURE_RIGHT_WITH_LEGEND = 0.875
FIGURE_WSPACE = 0.045
FIGURE_HSPACE_MULTIROW = 0.56

# ---------------------------------------------------------------------
# Temporal stability
# ---------------------------------------------------------------------
MIN_VALID_MONTHS_FOR_STABILITY = 18
STABILITY_SIGN_THRESHOLD = 0.0

# ---------------------------------------------------------------------
# Optional local/ego maps
# ---------------------------------------------------------------------
RUN_EGO_NETWORK_MAPS = True
EGO_DISTRICT_IDS = []  # Populate manually, e.g. ["2807901", "..."].
AUTO_SELECT_EGO_DISTRICTS = True
AUTO_EGO_PER_ROLE = 3
EGO_TOP_PARTNERS = 35
EGO_BUFFER_KM = 120

# ---------------------------------------------------------------------
# Exports
# ---------------------------------------------------------------------
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SHOW_FIGURES = True

print("Notebook-01 input stage:", portable_path_label(FORMAL_STAGE_DIR))
print("Notebook-02 output stage:", portable_path_label(MAPPING_STAGE_DIR))


## 3. Imports and general utilities

In [ ]:
import hashlib
import json
import math
import re
import unicodedata
import warnings
from datetime import datetime

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import shapely

from matplotlib.collections import LineCollection
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shapely.geometry import LineString

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 260)

mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.titlesize": 13,
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
})

warnings.filterwarnings("ignore", category=UserWarning)


def canonical_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    text = unicodedata.normalize("NFKC", text)
    return text if text else None


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required input does not exist: {path}\n"
            "Run Notebook 01 successfully before this mapping notebook."
        )
    return path


def require_columns(frame, columns, label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def atomic_to_csv(frame, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)


def atomic_to_parquet(frame, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary, index=False, compression="zstd")
    temporary.replace(path)


def save_figure(figure, stem, directory=FIGURE_MAIN_DIR):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    saved = []
    if SAVE_PNG:
        path = directory / f"{stem}.png"
        figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        saved.append(path)
    if SAVE_PDF:
        path = directory / f"{stem}.pdf"
        figure.savefig(path, bbox_inches="tight")
        saved.append(path)
    if SAVE_SVG:
        path = directory / f"{stem}.svg"
        figure.savefig(path, bbox_inches="tight")
        saved.append(path)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)
    return saved


def add_north_arrow(axis, x=0.955, y=0.94, size=0.07):
    axis.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - size),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=11,
        arrowprops=dict(arrowstyle="-|>", linewidth=1.0, color="black"),
    )


def nice_scale_length(width_metres):
    target = width_metres / 5
    magnitude = 10 ** math.floor(math.log10(max(target, 1)))
    candidates = np.array([1, 2, 5, 10]) * magnitude
    return float(candidates[np.argmin(np.abs(candidates - target))])


def add_scale_bar(axis, location=(0.07, 0.055), linewidth=2.0):
    xmin, xmax = axis.get_xlim()
    ymin, ymax = axis.get_ylim()
    width = xmax - xmin
    height = ymax - ymin
    length = nice_scale_length(width)
    x0 = xmin + location[0] * width
    y0 = ymin + location[1] * height
    axis.plot([x0, x0 + length], [y0, y0], color="black", linewidth=linewidth)
    axis.plot([x0, x0], [y0 - 0.006 * height, y0 + 0.006 * height], color="black", linewidth=1)
    axis.plot([x0 + length, x0 + length], [y0 - 0.006 * height, y0 + 0.006 * height], color="black", linewidth=1)
    label = f"{int(round(length / 1000))} km"
    axis.text(x0 + length / 2, y0 + 0.012 * height, label, ha="center", va="bottom", fontsize=8)


def finish_map_axis(
    axis,
    title=None,
    add_cartography=True,
    has_canary_inset=False,
):
    axis.set_axis_off()
    if title:
        axis.set_title(title, pad=8)
    if add_cartography:
        add_north_arrow(axis)
        scale_location = (
            MAIN_SCALE_BAR_LOCATION
            if has_canary_inset and ENABLE_CANARY_INSET
            else (0.07, 0.055)
        )
        add_scale_bar(axis, location=scale_location)


def robust_limits(values, lower=0.01, upper=0.99):
    series = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if series.empty:
        return (0.0, 1.0)
    lo, hi = series.quantile([lower, upper]).tolist()
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(series.min()), float(series.max())
    if lo == hi:
        hi = lo + 1.0
    return float(lo), float(hi)


def symmetric_limit(values, quantile=DIVERGING_LIMIT_QUANTILE):
    series = pd.to_numeric(pd.Series(values), errors="coerce").dropna().abs()
    if series.empty:
        return 1.0
    limit = float(series.quantile(quantile))
    if not np.isfinite(limit) or limit <= 0:
        limit = float(series.max()) if len(series) else 1.0
    return limit if limit > 0 else 1.0




def atomic_write_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def analysis_signature(paths):
    payload = []
    for path in paths:
        path = Path(path)
        payload.append({
            "path": str(path),
            "size": path.stat().st_size if path.exists() else None,
            "mtime": path.stat().st_mtime if path.exists() else None,
        })
    payload.append({
        "mapping_run_tag": MAPPING_RUN_TAG,
        "formal_run_tag": FORMAL_RUN_TAG,
        "map_crs": MAP_CRS,
        "flow_filter_mode": FLOW_FILTER_MODE,
        "flow_render_mode": FLOW_RENDER_MODE,
        "national_network_measure": NATIONAL_NETWORK_MEASURE,
        "national_top_n": NATIONAL_TOP_N,
        "difference_top_n": DIFFERENCE_TOP_N,
    })
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]

## 4. Load Notebook-01 outputs

In [ ]:
# ---------------------------------------------------------------------
# Notebook-01 version and provenance preflight
# ---------------------------------------------------------------------
require_file(FORMAL_MANIFEST_FILE)

with FORMAL_MANIFEST_FILE.open(
    "r",
    encoding="utf-8",
) as handle:
    formal_manifest = json.load(handle)

actual_formal_run_tag = formal_manifest.get("run_tag")
if actual_formal_run_tag != FORMAL_RUN_TAG:
    raise RuntimeError(
        "Notebook 02 expected Notebook-01 run tag "
        f"{FORMAL_RUN_TAG!r}, but the manifest reports "
        f"{actual_formal_run_tag!r}. Run the dated v4.4 Notebook 01 "
        "completely before running this notebook."
    )

print("Validated Notebook-01 run tag:", actual_formal_run_tag)


for required_path in [
    PERIOD_LAYER_EDGE_FILE,
    DISTRICT_PERIOD_FILE,
    DISTRICT_MONTH_FILE,
    MARGINAL_EFFECT_FILE,
    CENTROID_LOOKUP_FILE,
]:
    require_file(required_path)

period_layer_edges = pd.read_parquet(
    PERIOD_LAYER_EDGE_FILE
)
district_period = pd.read_parquet(
    DISTRICT_PERIOD_FILE
)
district_month = pd.read_parquet(
    DISTRICT_MONTH_FILE
)
marginal_effects = pd.read_parquet(
    MARGINAL_EFFECT_FILE
)
centroids = pd.read_parquet(
    CENTROID_LOOKUP_FILE
)

pairwise_similarity = (
    pd.read_csv(PAIRWISE_SIMILARITY_FILE)
    if PAIRWISE_SIMILARITY_FILE.exists()
    else pd.DataFrame()
)

require_columns(
    period_layer_edges,
    [
        "recurrence_band",
        "origin",
        "destination",
        "weight",
        "weight_share",
        "hybrid_work_intensity",
    ],
    "period_layer_edges",
)

require_columns(
    district_period,
    [
        "district_id",
        "role",
        "hybrid_intensity_all_flows",
        "hybrid_intensity_external",
        "partner_coverage",
        "partner_diversity",
        "mean_external_distance_km",
        "settlement_urbanity",
        "onsite_network_position",
        "has_any_flow",
        "has_external_relation",
    ],
    "district_period",
)

require_columns(
    centroids,
    ["district_id", "x", "y"],
    "centroid lookup",
)

require_columns(
    marginal_effects,
    [
        "district_id",
        "role",
        "outcome",
        "intensity_effect_per_0_1_pct",
    ],
    "marginal_effects",
)

period_layer_edges["origin"] = period_layer_edges[
    "origin"
].map(canonical_id)
period_layer_edges["destination"] = period_layer_edges[
    "destination"
].map(canonical_id)

for frame in [
    district_period,
    district_month,
    marginal_effects,
    centroids,
]:
    frame["district_id"] = frame[
        "district_id"
    ].map(canonical_id)

formal_district_ids = set(
    centroids["district_id"].dropna()
)
formal_district_ids.update(
    period_layer_edges["origin"].dropna()
)
formal_district_ids.update(
    period_layer_edges["destination"].dropna()
)
formal_district_ids.update(
    district_period["district_id"].dropna()
)

formal_district_ids = {
    district_id
    for district_id in formal_district_ids
    if re.match(
        SPAIN_DISTRICT_ID_PATTERN,
        district_id or "",
    )
}

period_layer_edges = period_layer_edges.loc[
    period_layer_edges["origin"].isin(
        formal_district_ids
    )
    & period_layer_edges["destination"].isin(
        formal_district_ids
    )
].copy()

district_period = district_period.loc[
    district_period["district_id"].isin(
        formal_district_ids
    )
].copy()

district_month = district_month.loc[
    district_month["district_id"].isin(
        formal_district_ids
    )
].copy()

marginal_effects = marginal_effects.loc[
    marginal_effects["district_id"].isin(
        formal_district_ids
    )
].copy()

period_edges = period_layer_edges
period_premiums = district_period
monthly_premiums = district_month

centroid_lookup = (
    centroids.drop_duplicates(
        "district_id"
    ).copy()
)

centroid_gdf = gpd.GeoDataFrame(
    centroid_lookup.copy(),
    geometry=gpd.points_from_xy(
        centroid_lookup["x"],
        centroid_lookup["y"],
    ),
    crs=MAP_CRS,
)


def is_spanish_district_id(value):
    text = canonical_id(value)
    return bool(
        text
        and re.match(
            SPAIN_DISTRICT_ID_PATTERN,
            text,
        )
    )


scope_diagnostics = pd.DataFrame([
    {
        "dataset": "period_layer_edges",
        "spain_rows": len(period_layer_edges),
    },
    {
        "dataset": "district_period",
        "spain_rows": len(district_period),
    },
    {
        "dataset": "district_month",
        "spain_rows": len(district_month),
    },
    {
        "dataset": "marginal_effects",
        "spain_rows": len(marginal_effects),
    },
    {
        "dataset": "centroids",
        "spain_rows": len(centroid_lookup),
    },
])

atomic_to_csv(
    scope_diagnostics,
    LOG_DIR / "spain_only_data_scope_diagnostics.csv",
)

print("Period layer edges:", len(period_layer_edges))
print(
    "Period district-role observations:",
    len(district_period),
)
print(
    "Marginal-effect observations:",
    len(marginal_effects),
)
print(
    "Formal Spanish district IDs:",
    len(formal_district_ids),
)


# ---------------------------------------------------------------------
# Notebook-01 provenance and GHSL audit
# ---------------------------------------------------------------------
ghs_profile_audit = (
    pd.read_csv(GHS_PROFILE_AUDIT_FILE)
    if GHS_PROFILE_AUDIT_FILE.exists()
    else pd.DataFrame()
)

period_ghs_audit = (
    pd.read_csv(PERIOD_GHS_AUDIT_FILE)
    if PERIOD_GHS_AUDIT_FILE.exists()
    else pd.DataFrame()
)

ghs_match_share = np.nan

if (
    not period_ghs_audit.empty
    and "unique_district_match_share"
    in period_ghs_audit.columns
):
    ghs_match_share = float(
        period_ghs_audit.loc[
            0,
            "unique_district_match_share",
        ]
    )

    if (
        np.isfinite(ghs_match_share)
        and ghs_match_share < 0.90
    ):
        raise RuntimeError(
            "The period-level GHSL district match recorded by Notebook 01 "
            f"is below 90%: {ghs_match_share:.2%}"
        )

print(
    "Source analysis notebook:",
    formal_manifest.get(
        "notebook",
        "not recorded",
    ),
)
print(
    "Source analysis signature:",
    formal_manifest.get(
        "analysis_signature",
        "not recorded",
    ),
)
print(
    "GHSL profile:",
    portable_path_label(
        formal_manifest.get(
            "ghs_profile_path",
            "not recorded",
        )
    ),
)

if np.isfinite(ghs_match_share):
    print(
        "Period-level GHSL district match:",
        f"{ghs_match_share:.2%}",
    )


## 5. Discover district polygons and optional province boundaries

In [ ]:
def spatial_candidates(
    search_roots,
    suffixes=(".shp", ".gpkg", ".geojson", ".parquet"),
):
    candidates = []
    seen = set()
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for suffix in suffixes:
            for path in root.rglob(f"*{suffix}"):
                if path not in seen:
                    seen.add(path)
                    candidates.append(path)
    return candidates


def score_id_fields(frame, target_ids):
    target_ids = set(target_ids)
    geometry_name = frame.geometry.name
    rows = []

    for column in frame.columns:
        if column == geometry_name:
            continue

        values = frame[column].map(canonical_id)
        unique_values = set(values.dropna().unique())
        intersection = unique_values & target_ids

        rows.append(
            {
                "field": column,
                "match_count": len(intersection),
                "target_match_share": (
                    len(intersection) / len(target_ids)
                    if target_ids
                    else np.nan
                ),
                "field_unique_match_share": (
                    len(intersection) / len(unique_values)
                    if unique_values
                    else np.nan
                ),
            }
        )

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "target_match_share",
                "field_unique_match_share",
                "match_count",
            ],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )


def read_spatial_file(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return gpd.read_parquet(path)
    return gpd.read_file(path)


FORMAL_SPANISH_DISTRICT_IDS = set(
    pd.concat(
        [
            period_premiums["district_id"],
            period_edges["origin"],
            period_edges["destination"],
        ],
        ignore_index=True,
    )
    .map(canonical_id)
    .dropna()
    .unique()
)


def select_district_polygon_file():
    roots = [
        BASEMAP_DIR,
        REFERENCE_DIR,
        PROJECT_ROOT / "03_processed",
    ]
    candidates = spatial_candidates(roots)
    scored = []

    preferred_lookup = {
        name.lower(): rank
        for rank, name in enumerate(
            DISTRICT_POLYGON_PREFERRED_NAMES
        )
    }

    for path in candidates:
        try:
            frame = read_spatial_file(path)
        except Exception:
            continue

        if frame.empty or frame.crs is None:
            continue

        polygon_share = frame.geom_type.isin(
            ["Polygon", "MultiPolygon"]
        ).mean()
        if polygon_share < 0.80:
            continue

        scores = score_id_fields(
            frame,
            FORMAL_SPANISH_DISTRICT_IDS,
        )
        if scores.empty:
            continue

        best = scores.iloc[0]
        scored.append(
            {
                "path": path,
                "id_field": best["field"],
                "match_count": int(best["match_count"]),
                "target_match_share": float(
                    best["target_match_share"]
                ),
                "preferred_rank": preferred_lookup.get(
                    path.name.lower(),
                    999,
                ),
                "feature_count": len(frame),
            }
        )

    if not scored:
        raise FileNotFoundError(
            "No polygon file matching the formal Spanish district IDs "
            "was found."
        )

    scored = (
        pd.DataFrame(scored)
        .sort_values(
            [
                "target_match_share",
                "preferred_rank",
                "match_count",
            ],
            ascending=[False, True, False],
        )
        .reset_index(drop=True)
    )
    return scored.iloc[0], scored


district_choice, district_candidate_scores = (
    select_district_polygon_file()
)
district_polygon_path = Path(
    district_choice["path"]
)
district_id_field = district_choice[
    "id_field"
]

districts_raw = read_spatial_file(
    district_polygon_path
)
districts_raw["district_id"] = districts_raw[
    district_id_field
].map(canonical_id)

# Full Spanish cartographic universe, independent of network-table coverage.
districts_raw["_country_scope"] = np.select(
    [
        districts_raw["district_id"].map(is_spanish_district_id),
        districts_raw["district_id"].str.startswith("FR", na=False),
        districts_raw["district_id"].str.startswith("PT", na=False),
    ],
    ["Spain", "France external zone", "Portugal external zone"],
    default="Other external or invalid zone",
)
districts = districts_raw.loc[
    (districts_raw["_country_scope"] == "Spain")
    & districts_raw.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()
districts = districts.drop_duplicates("district_id").to_crs(MAP_CRS)
districts["_in_formal_network"] = districts["district_id"].isin(
    FORMAL_SPANISH_DISTRICT_IDS
)
MASTER_SPANISH_DISTRICT_IDS = set(districts["district_id"])

formal_ids_with_polygon = FORMAL_SPANISH_DISTRICT_IDS & MASTER_SPANISH_DISTRICT_IDS
match_share = (
    len(formal_ids_with_polygon) / len(FORMAL_SPANISH_DISTRICT_IDS)
    if FORMAL_SPANISH_DISTRICT_IDS else np.nan
)
if FAIL_ON_COVERAGE_ERROR and np.isfinite(match_share) and match_share < MIN_FORMAL_ID_POLYGON_COVERAGE:
    raise RuntimeError(f"Formal Spanish ID polygon coverage is only {match_share:.3%}.")

country_scope_counts = (
    districts_raw["_country_scope"].value_counts(dropna=False)
    .rename_axis("country_scope").reset_index(name="polygon_count")
)
polygon_filter_diagnostics = pd.DataFrame([{
    "raw_polygon_features": len(districts_raw),
    "master_spanish_polygons": len(districts),
    "formal_spanish_network_ids": len(FORMAL_SPANISH_DISTRICT_IDS),
    "formal_ids_with_polygon": len(formal_ids_with_polygon),
    "formal_id_polygon_coverage": match_share,
    "spanish_polygons_outside_formal_network": int((~districts["_in_formal_network"]).sum()),
    "fr_polygons_removed": int((districts_raw["_country_scope"] == "France external zone").sum()),
    "pt_polygons_removed": int((districts_raw["_country_scope"] == "Portugal external zone").sum()),
}])
atomic_to_csv(
    district_candidate_scores.assign(path=lambda frame: frame["path"].astype(str)),
    LOG_DIR / "district_polygon_candidate_scores.csv",
)
atomic_to_csv(country_scope_counts, LOG_DIR / "polygon_country_scope_counts.csv")
atomic_to_csv(polygon_filter_diagnostics, LOG_DIR / "spain_master_geography_diagnostics.csv")

print("Master Spanish polygons:", len(districts))
print("French polygons removed:", polygon_filter_diagnostics.loc[0, "fr_polygons_removed"])
print("Portuguese polygons removed:", polygon_filter_diagnostics.loc[0, "pt_polygons_removed"])
print(f"Formal Spanish ID polygon coverage: {match_share:.2%}")
display(country_scope_counts)
display(polygon_filter_diagnostics)
def province_code_from_district_id(district_id):
    text = canonical_id(district_id)
    if text is None:
        return None

    digits = "".join(
        character
        for character in text
        if character.isdigit()
    )
    if len(digits) < PROVINCE_CODE_DIGITS:
        return None

    return digits[:PROVINCE_CODE_DIGITS]


def _repair_polygonal_geometries(frame):
    repaired = frame.copy()

    invalid_before = int(
        (~repaired.geometry.is_valid).sum()
    )

    # GeoPandas/Shapely 2 path.
    try:
        repaired["geometry"] = (
            repaired.geometry.make_valid()
        )
    except Exception:
        repaired["geometry"] = (
            repaired.geometry.apply(
                lambda geometry: (
                    shapely.make_valid(geometry)
                    if geometry is not None
                    else None
                )
            )
        )

    # make_valid may return MultiPolygons or GeometryCollections.
    repaired = repaired.explode(
        index_parts=False,
        ignore_index=True,
    )
    repaired = repaired.loc[
        repaired.geometry.notna()
        & ~repaired.geometry.is_empty
        & repaired.geom_type.isin(
            ["Polygon", "MultiPolygon"]
        )
    ].copy()

    invalid_after_make_valid = int(
        (~repaired.geometry.is_valid).sum()
    )

    if invalid_after_make_valid:
        repaired.loc[
            ~repaired.geometry.is_valid,
            "geometry",
        ] = (
            repaired.loc[
                ~repaired.geometry.is_valid
            ].geometry.buffer(0)
        )

    repaired = repaired.loc[
        repaired.geometry.notna()
        & ~repaired.geometry.is_empty
        & repaired.geom_type.isin(
            ["Polygon", "MultiPolygon"]
        )
    ].copy()

    invalid_after_buffer = int(
        (~repaired.geometry.is_valid).sum()
    )

    diagnostics = {
        "invalid_before_repair": invalid_before,
        "invalid_after_make_valid": (
            invalid_after_make_valid
        ),
        "invalid_after_buffer_zero": (
            invalid_after_buffer
        ),
        "polygon_parts_after_repair": len(
            repaired
        ),
    }
    return repaired, diagnostics


def derive_province_boundaries(district_frame):
    if not DERIVE_PROVINCES_FROM_DISTRICT_IDS:
        return None, np.nan, {
            "status": "disabled",
        }

    working = district_frame[
        ["district_id", "geometry"]
    ].copy()
    working["_province_code"] = working[
        "district_id"
    ].map(province_code_from_district_id)

    prefix_coverage = working[
        "_province_code"
    ].notna().mean()

    if prefix_coverage < MIN_PROVINCE_PREFIX_COVERAGE:
        return None, prefix_coverage, {
            "status": "insufficient_prefix_coverage",
        }

    working = working.dropna(
        subset=["_province_code"]
    ).copy()

    repaired, diagnostics = (
        _repair_polygonal_geometries(
            working
        )
    )

    try:
        derived = repaired.dissolve(
            by="_province_code",
            as_index=False,
        )
        diagnostics["status"] = "success"
        diagnostics["province_count"] = len(
            derived
        )

    except Exception as first_error:
        # Precision snapping can resolve very small side-location
        # conflicts that remain after make_valid/buffer(0).
        diagnostics["first_dissolve_error"] = (
            repr(first_error)
        )

        try:
            snapped = repaired.copy()
            snapped["geometry"] = (
                snapped.geometry.apply(
                    lambda geometry: (
                        shapely.set_precision(
                            geometry,
                            grid_size=1.0,
                        )
                        if geometry is not None
                        else None
                    )
                )
            )
            snapped = snapped.loc[
                snapped.geometry.notna()
                & ~snapped.geometry.is_empty
                & snapped.geom_type.isin(
                    ["Polygon", "MultiPolygon"]
                )
            ].copy()

            derived = snapped.dissolve(
                by="_province_code",
                as_index=False,
            )
            diagnostics["status"] = (
                "success_after_precision_snap"
            )
            diagnostics["province_count"] = len(
                derived
            )

        except Exception as second_error:
            diagnostics["status"] = (
                "omitted_after_failed_repair"
            )
            diagnostics["second_dissolve_error"] = (
                repr(second_error)
            )

            print(
                "Province-boundary dissolve failed after "
                "geometry repair. Province overlays will be "
                "omitted; district maps will continue."
            )
            print(
                "First dissolve error:",
                repr(first_error),
            )
            print(
                "Second dissolve error:",
                repr(second_error),
            )
            return None, prefix_coverage, diagnostics

    derived = gpd.GeoDataFrame(
        derived,
        geometry="geometry",
        crs=district_frame.crs,
    )
    return derived, prefix_coverage, diagnostics

(
    provinces,
    province_prefix_coverage,
    province_geometry_diagnostics,
) = derive_province_boundaries(
    districts
)
province_polygon_path = None

province_geometry_log = pd.DataFrame(
    [province_geometry_diagnostics]
)
atomic_to_csv(
    province_geometry_log,
    LOG_DIR / "province_geometry_repair_diagnostics.csv",
)

if provinces is not None:
    print(
        "Province boundaries derived from the filtered Spanish "
        "district polygons."
    )
    print(
        f"Province-prefix coverage: "
        f"{province_prefix_coverage:.2%}"
    )
    print(
        "Province geometry status:",
        province_geometry_diagnostics.get(
            "status"
        ),
    )
else:
    print(
        "Province overlays omitted. District maps will continue "
        "without province outlines."
    )
    print(
        "Province geometry status:",
        province_geometry_diagnostics.get(
            "status"
        ),
    )

display(province_geometry_log)

## 5A. Identify the Canary Islands and establish fixed map extents

The inset classification is geographic rather than dependent on a particular
district-code prefix. Representative points are transformed to longitude and
latitude, then classified using the configured thresholds.

In [ ]:
def padded_bounds(geodata, padding_share):
    if geodata is None or geodata.empty:
        return None
    xmin, ymin, xmax, ymax = geodata.total_bounds
    width = max(xmax - xmin, 1.0)
    height = max(ymax - ymin, 1.0)
    return (
        xmin - width * padding_share,
        ymin - height * padding_share,
        xmax + width * padding_share,
        ymax + height * padding_share,
    )


def representative_lon_lat(geodata):
    geographic = geodata.to_crs("EPSG:4326")
    points = geographic.geometry.representative_point()
    return pd.DataFrame(
        {
            "longitude": points.x.to_numpy(),
            "latitude": points.y.to_numpy(),
        },
        index=geodata.index,
    )


district_location = representative_lon_lat(districts)
districts["_representative_longitude"] = district_location["longitude"]
districts["_representative_latitude"] = district_location["latitude"]
districts["_is_canary"] = (
    (districts["_representative_longitude"] < CANARY_LONGITUDE_MAX)
    & (districts["_representative_latitude"] < CANARY_LATITUDE_MAX)
)

canary_districts = districts.loc[districts["_is_canary"]].copy()
main_map_districts = districts.loc[~districts["_is_canary"]].copy()

if ENABLE_CANARY_INSET and canary_districts.empty:
    raise RuntimeError(
        "The Canary inset is enabled, but no district polygons were "
        "identified by the configured geographic thresholds."
    )

CANARY_DISTRICT_IDS = set(canary_districts["district_id"].dropna())
MAIN_MAP_DISTRICT_IDS = set(main_map_districts["district_id"].dropna())

MAIN_MAP_BOUNDS = padded_bounds(main_map_districts, MAIN_EXTENT_PADDING_SHARE)
CANARY_BOUNDS = padded_bounds(canary_districts, CANARY_EXTENT_PADDING_SHARE)

if provinces is not None:
    province_location = representative_lon_lat(provinces)
    provinces["_representative_longitude"] = province_location["longitude"]
    provinces["_representative_latitude"] = province_location["latitude"]
    provinces["_is_canary"] = (
        (provinces["_representative_longitude"] < CANARY_LONGITUDE_MAX)
        & (provinces["_representative_latitude"] < CANARY_LATITUDE_MAX)
    )
    canary_provinces = provinces.loc[provinces["_is_canary"]].copy()
    main_map_provinces = provinces.loc[~provinces["_is_canary"]].copy()
else:
    canary_provinces = None
    main_map_provinces = None

territory_diagnostics = pd.DataFrame(
    [
        {
            "territory": "Main map",
            "district_count": len(main_map_districts),
            "minimum_longitude": main_map_districts["_representative_longitude"].min(),
            "maximum_longitude": main_map_districts["_representative_longitude"].max(),
            "minimum_latitude": main_map_districts["_representative_latitude"].min(),
            "maximum_latitude": main_map_districts["_representative_latitude"].max(),
        },
        {
            "territory": "Canary Islands inset",
            "district_count": len(canary_districts),
            "minimum_longitude": canary_districts["_representative_longitude"].min(),
            "maximum_longitude": canary_districts["_representative_longitude"].max(),
            "minimum_latitude": canary_districts["_representative_latitude"].min(),
            "maximum_latitude": canary_districts["_representative_latitude"].max(),
        },
    ]
)

canary_district_lookup = canary_districts[
    [
        "district_id",
        "_representative_longitude",
        "_representative_latitude",
    ]
].copy()

atomic_to_csv(
    territory_diagnostics,
    LOG_DIR / "national_map_territory_diagnostics.csv",
)
atomic_to_csv(
    canary_district_lookup,
    LOG_DIR / "canary_district_lookup.csv",
)

print("Main-map districts:", len(main_map_districts))
print("Canary inset districts:", len(canary_districts))
print("Main-map bounds:", MAIN_MAP_BOUNDS)
print("Canary inset bounds:", CANARY_BOUNDS)
display(territory_diagnostics)
display(canary_district_lookup.head(20))

def resolved_canary_inset_bounds():
    if MAIN_MAP_BOUNDS is None:
        return (
            CANARY_INSET_BASE_LEFT,
            CANARY_INSET_BASE_BOTTOM,
            CANARY_INSET_WIDTH,
            CANARY_INSET_HEIGHT,
        )

    xmin, ymin, xmax, ymax = MAIN_MAP_BOUNDS
    main_width_metres = max(xmax - xmin, 1.0)
    main_height_metres = max(ymax - ymin, 1.0)

    shift_x_fraction = (
        CANARY_INSET_SHIFT_X_KM
        * 1000.0
        / main_width_metres
    )
    shift_y_fraction = (
        CANARY_INSET_SHIFT_Y_KM
        * 1000.0
        / main_height_metres
    )

    return (
        CANARY_INSET_BASE_LEFT + shift_x_fraction,
        CANARY_INSET_BASE_BOTTOM + shift_y_fraction,
        CANARY_INSET_WIDTH,
        CANARY_INSET_HEIGHT,
    )


RESOLVED_CANARY_INSET_BOUNDS = (
    resolved_canary_inset_bounds()
)

inset_position_diagnostics = pd.DataFrame(
    [
        {
            "base_left": CANARY_INSET_BASE_LEFT,
            "base_bottom": CANARY_INSET_BASE_BOTTOM,
            "width": CANARY_INSET_WIDTH,
            "height": CANARY_INSET_HEIGHT,
            "shift_x_km": CANARY_INSET_SHIFT_X_KM,
            "shift_y_km": CANARY_INSET_SHIFT_Y_KM,
            "resolved_left": RESOLVED_CANARY_INSET_BOUNDS[0],
            "resolved_bottom": RESOLVED_CANARY_INSET_BOUNDS[1],
        }
    ]
)
atomic_to_csv(
    inset_position_diagnostics,
    LOG_DIR / "canary_inset_position_diagnostics.csv",
)

print(
    "Resolved Canary inset axes bounds:",
    RESOLVED_CANARY_INSET_BOUNDS,
)
display(inset_position_diagnostics)

## 5B. Complete Spanish district coverage and run QA

In [ ]:
role_grid = pd.MultiIndex.from_product(
    [
        sorted(
            districts["district_id"]
            .dropna()
            .unique()
        ),
        ["residential", "employment"],
    ],
    names=["district_id", "role"],
).to_frame(index=False)

district_period_display = role_grid.merge(
    district_period,
    on=["district_id", "role"],
    how="left",
    validate="one_to_one",
)

district_period_display[
    "has_formal_observation"
] = district_period_display[
    "hybrid_intensity_all_flows"
].notna()

coverage_qa = pd.DataFrame([
    {
        "measure": column,
        "valid_district_role_records": int(
            district_period_display[
                column
            ].notna().sum()
        ),
        "coverage_share": float(
            district_period_display[
                column
            ].notna().mean()
        ),
    }
    for column in [
        "hybrid_intensity_all_flows",
        "hybrid_intensity_external",
        "partner_coverage",
        "partner_diversity",
        "mean_external_distance_km",
        "settlement_urbanity",
        "onsite_network_position",
    ]
])

atomic_to_csv(
    coverage_qa,
    LOG_DIR / "district_map_value_coverage.csv",
)

display(coverage_qa)

period_premiums_display = district_period_display


## 6. Build line geometry and shared cartographic helpers

In [ ]:
centroid_xy = centroid_gdf.set_index("district_id")[["x", "y"]]


def edges_to_geodataframe(edges):
    working = edges.copy()

    working["origin_x"] = working["origin"].map(
        centroid_xy["x"]
    )
    working["origin_y"] = working["origin"].map(
        centroid_xy["y"]
    )
    working["destination_x"] = working["destination"].map(
        centroid_xy["x"]
    )
    working["destination_y"] = working["destination"].map(
        centroid_xy["y"]
    )

    working = working.dropna(
        subset=[
            "origin_x",
            "origin_y",
            "destination_x",
            "destination_y",
        ]
    ).copy()

    working["geometry"] = [
        LineString(
            [
                (origin_x, origin_y),
                (destination_x, destination_y),
            ]
        )
        for origin_x, origin_y, destination_x, destination_y in zip(
            working["origin_x"],
            working["origin_y"],
            working["destination_x"],
            working["destination_y"],
        )
    ]

    origin_is_canary = working["origin"].isin(
        CANARY_DISTRICT_IDS
    )
    destination_is_canary = working["destination"].isin(
        CANARY_DISTRICT_IDS
    )

    working["_edge_map_region"] = np.select(
        [
            (~origin_is_canary)
            & (~destination_is_canary),
            origin_is_canary
            & destination_is_canary,
        ],
        [
            "main",
            "canary",
        ],
        default="cross_window",
    )

    return gpd.GeoDataFrame(
        working,
        geometry="geometry",
        crs=MAP_CRS,
    )


edge_gdf = edges_to_geodataframe(
    period_edges
)

coordinate_coverage = (
    len(edge_gdf) / len(period_edges)
    if len(period_edges)
    else np.nan
)

print(
    f"Edge coordinate coverage: "
    f"{coordinate_coverage:.2%}"
)
display(
    edge_gdf["_edge_map_region"]
    .value_counts(
        dropna=False
    )
)


FLOW_SCALE_LOG = []
MAP_NORM_LOG = []


def _positive_external_edges(
    frame,
    weight_column,
):
    working = frame.copy()

    weights = pd.to_numeric(
        working[weight_column],
        errors="coerce",
    )

    keep = (
        weights > FLOW_MIN_WEIGHT
    )

    if FLOW_MAP_EXCLUDE_SELF_LOOPS:
        keep &= (
            working["origin"]
            != working["destination"]
        )

    working = working.loc[
        keep
    ].copy()

    working[weight_column] = pd.to_numeric(
        working[weight_column],
        errors="coerce",
    )

    return working


def filter_edges(
    frame,
    weight_column,
    mode=FLOW_FILTER_MODE,
    top_n=NATIONAL_TOP_N,
    quantile=EDGE_WEIGHT_QUANTILE,
    cumulative_share=CUMULATIVE_EDGE_MASS,
    global_top_n=FLOW_COVERAGE_GLOBAL_TOP_N,
    top_k_out=FLOW_COVERAGE_TOP_K_OUT,
    top_k_in=FLOW_COVERAGE_TOP_K_IN,
    max_edges=FLOW_COVERAGE_MAX_EDGES,
):
    working = _positive_external_edges(
        frame,
        weight_column,
    ).sort_values(
        weight_column,
        ascending=False,
    )

    if working.empty:
        return working

    if mode == "all":
        return working.copy()

    if mode == "top_n":
        return working.head(
            int(top_n)
        ).copy()

    if mode == "weight_quantile":
        threshold = working[
            weight_column
        ].quantile(
            quantile
        )
        return working.loc[
            working[weight_column]
            >= threshold
        ].copy()

    if mode == "cumulative_share":
        total = working[
            weight_column
        ].sum()

        if total <= 0:
            return working.iloc[
                0:0
            ].copy()

        working["_share"] = (
            working[weight_column]
            / total
        )
        working["_cumulative"] = (
            working["_share"]
            .cumsum()
        )

        keep = (
            working["_cumulative"]
            .shift(
                fill_value=0
            )
            < cumulative_share
        )

        return (
            working.loc[
                keep
            ]
            .drop(
                columns=[
                    "_share",
                    "_cumulative",
                ]
            )
            .copy()
        )

    if mode == "coverage":
        global_edges = working.head(
            int(global_top_n)
        )

        strongest_out = (
            working.groupby(
                "origin",
                group_keys=False,
            )
            .head(
                int(top_k_out)
            )
        )

        strongest_in = (
            working.groupby(
                "destination",
                group_keys=False,
            )
            .head(
                int(top_k_in)
            )
        )

        selected = pd.concat(
            [
                global_edges,
                strongest_out,
                strongest_in,
            ],
            ignore_index=False,
        )

        selected = (
            selected.drop_duplicates(
                subset=[
                    "origin",
                    "destination",
                ]
            )
            .sort_values(
                weight_column,
                ascending=False,
            )
        )

        if max_edges is not None:
            selected = selected.head(
                int(max_edges)
            )

        return selected.copy()

    raise ValueError(
        "Unknown FLOW_FILTER_MODE. "
        "Use all, top_n, weight_quantile, "
        "cumulative_share or coverage."
    )


def combined_positive_values(
    frame_column_pairs,
    exclude_cross_window=True,
):
    parts = []

    for frame, column in frame_column_pairs:
        working = frame

        if (
            exclude_cross_window
            and "_edge_map_region"
            in working.columns
        ):
            working = working.loc[
                working["_edge_map_region"]
                != "cross_window"
            ]

        part = _positive_external_edges(
            working,
            column,
        )

        if not part.empty:
            parts.append(
                pd.to_numeric(
                    part[column],
                    errors="coerce",
                )
            )

    if not parts:
        return pd.Series(
            dtype=float
        )

    return pd.concat(
        parts,
        ignore_index=True,
    ).dropna()


def build_flow_scale(
    values,
    label,
    measure_type,
    scale_name=None,
    minimum=BACKBONE_MIN_WIDTH,
    maximum=BACKBONE_MAX_WIDTH,
    cap_quantile=FLOW_LINEWIDTH_CAP_QUANTILE,
    backbone_quantile=FLOW_BACKBONE_QUANTILE,
    legend_quantiles=FLOW_LEGEND_QUANTILES,
):
    series = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    )

    series = series.loc[
        np.isfinite(series)
        & (series > 0)
    ]

    if series.empty:
        series = pd.Series(
            [1.0]
        )

    cap = float(
        series.quantile(
            cap_quantile
        )
    )
    cap = max(
        cap,
        np.finfo(float).eps,
    )

    backbone_threshold = float(
        series.quantile(
            backbone_quantile
        )
    )

    legend_values = np.unique(
        series.quantile(
            list(
                legend_quantiles
            )
        ).to_numpy(
            dtype=float
        )
    )

    legend_values = legend_values[
        np.isfinite(
            legend_values
        )
        & (
            legend_values > 0
        )
    ]

    if len(legend_values) == 0:
        legend_values = np.array(
            [cap]
        )

    scale = {
        "name": (
            scale_name
            or label
        ),
        "label": label,
        "measure_type": measure_type,
        "minimum_width": float(
            minimum
        ),
        "maximum_width": float(
            maximum
        ),
        "cap": float(
            cap
        ),
        "backbone_threshold": float(
            backbone_threshold
        ),
        "legend_values": [
            float(value)
            for value in legend_values
        ],
        "cap_quantile": float(
            cap_quantile
        ),
        "backbone_quantile": float(
            backbone_quantile
        ),
    }

    return scale


def register_flow_scale(
    scale_name,
    scale,
):
    record = {
        "scale_name": scale_name,
        "label": scale["label"],
        "measure_type": scale[
            "measure_type"
        ],
        "minimum_width": scale[
            "minimum_width"
        ],
        "maximum_width": scale[
            "maximum_width"
        ],
        "cap": scale["cap"],
        "backbone_threshold": scale[
            "backbone_threshold"
        ],
        "legend_values": "|".join(
            _trim_fixed_decimal(value, decimals=10)
            for value in scale[
                "legend_values"
            ]
        ),
        "cap_quantile": scale[
            "cap_quantile"
        ],
        "backbone_quantile": scale[
            "backbone_quantile"
        ],
    }

    FLOW_SCALE_LOG.append(
        record
    )


def scaled_linewidth(
    values,
    scale=None,
    minimum=BACKBONE_MIN_WIDTH,
    maximum=BACKBONE_MAX_WIDTH,
    reference_cap=None,
):
    series = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).fillna(
        0.0
    )

    if scale is not None:
        minimum = scale[
            "minimum_width"
        ]
        maximum = scale[
            "maximum_width"
        ]
        reference_cap = scale[
            "cap"
        ]

    positive = series.loc[
        series > 0
    ]

    if positive.empty:
        return np.full(
            len(series),
            minimum,
        )

    if reference_cap is None:
        reference_cap = float(
            positive.quantile(
                FLOW_LINEWIDTH_CAP_QUANTILE
            )
        )

    reference_cap = max(
        float(
            reference_cap
        ),
        np.finfo(float).eps,
    )

    ratio = np.clip(
        series.to_numpy(
            dtype=float
        )
        / reference_cap,
        0.0,
        1.0,
    )

    if FLOW_LINEWIDTH_SCALE == "log1p":
        transformed = (
            np.log1p(
                FLOW_LOG_STRETCH
                * ratio
            )
            / np.log1p(
                FLOW_LOG_STRETCH
            )
        )

    elif FLOW_LINEWIDTH_SCALE == "sqrt":
        transformed = np.sqrt(
            ratio
        )

    elif FLOW_LINEWIDTH_SCALE == "linear":
        transformed = ratio

    else:
        raise ValueError(
            "FLOW_LINEWIDTH_SCALE must be "
            "log1p, sqrt or linear."
        )

    return (
        minimum
        + transformed
        * (
            maximum
            - minimum
        )
    )



def _trim_fixed_decimal(value, decimals=6):
    """Format a number in fixed notation and remove redundant zeros."""
    value = float(value)
    if not np.isfinite(value):
        return ""
    text = f"{value:.{int(decimals)}f}"
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    if text in {"-0", "-0.0", ""}:
        text = "0"
    return text


def _plain_tick_formatter(decimals=4):
    """Matplotlib formatter that never uses scientific notation."""
    return FuncFormatter(
        lambda value, position: _trim_fixed_decimal(value, decimals=decimals)
    )


def format_flow_value(
    value,
    measure_type,
):
    """Format flow legends in fixed decimal notation."""
    value = float(value)

    if measure_type == "raw_weight":
        if abs(value) >= 1:
            return f"{value:,.0f}"
        return _trim_fixed_decimal(
            value,
            decimals=FLOW_LEGEND_MAX_DECIMALS,
        )

    if measure_type == "share":
        percentage = value * 100.0

        if abs(percentage) >= 1:
            return f"{percentage:.1f}%"

        if abs(percentage) >= 0.01:
            return f"{percentage:.3f}%"

        decimals = (
            6
            if abs(percentage) >= 0.0001
            else FLOW_LEGEND_MAX_DECIMALS
        )
        formatted = _trim_fixed_decimal(
            percentage,
            decimals=decimals,
        )
        return f"{formatted}%"

    if measure_type == "share_difference":
        percentage_points = value * 100.0

        if abs(percentage_points) >= 0.1:
            return f"{percentage_points:.2f} pp"

        formatted = _trim_fixed_decimal(
            percentage_points,
            decimals=FLOW_LEGEND_MAX_DECIMALS,
        )
        return f"{formatted} pp"

    if measure_type == "percent":
        formatted = _trim_fixed_decimal(
            value,
            decimals=4,
        )
        return f"{formatted}%"

    return _trim_fixed_decimal(
        value,
        decimals=FLOW_LEGEND_MAX_DECIMALS,
    )



def flow_legend_handles(
    scale,
    colour="#555555",
    alpha=0.85,
):
    values = np.asarray(
        scale[
            "legend_values"
        ],
        dtype=float,
    )

    widths = scaled_linewidth(
        values,
        scale=scale,
    )

    return [
        Line2D(
            [0],
            [0],
            color=colour,
            linewidth=float(
                width
            ),
            alpha=alpha,
            label=format_flow_value(
                value,
                scale[
                    "measure_type"
                ],
            ),
        )
        for value, width in zip(
            values,
            widths,
        )
    ]


def add_figure_flow_legend(
    figure,
    scale,
    bbox_to_anchor,
    loc="center right",
    colour="#555555",
    title=None,
    alpha=0.85,
):
    legend = figure.legend(
        handles=flow_legend_handles(
            scale,
            colour=colour,
            alpha=alpha,
        ),
        title=(
            title
            or scale["label"]
        ),
        frameon=False,
        loc=loc,
        bbox_to_anchor=bbox_to_anchor,
        borderaxespad=0.0,
    )

    figure.add_artist(
        legend
    )

    return legend


def register_map_norm(
    norm_name,
    norm,
):
    MAP_NORM_LOG.append(
        {
            "norm_name": norm_name,
            "norm_type": type(
                norm
            ).__name__,
            "vmin": float(
                norm.vmin
            ),
            "vcenter": (
                float(
                    norm.vcenter
                )
                if hasattr(
                    norm,
                    "vcenter",
                )
                else np.nan
            ),
            "vmax": float(
                norm.vmax
            ),
        }
    )


def shared_sequential_norm(
    values,
):
    lo, hi = robust_limits(
        values,
        *CLIP_SEQUENTIAL_QUANTILES,
    )
    return Normalize(
        vmin=lo,
        vmax=hi,
    )


def shared_diverging_norm(
    values,
):
    limit = symmetric_limit(
        values
    )
    return TwoSlopeNorm(
        vmin=-limit,
        vcenter=0.0,
        vmax=limit,
    )


def apply_map_layout(
    figure,
    nrows=1,
    right=None,
    top=FIGURE_TOP,
    bottom=FIGURE_BOTTOM,
    left=FIGURE_LEFT,
    wspace=FIGURE_WSPACE,
    hspace=None,
):
    if right is None:
        right = (
            FIGURE_RIGHT_STANDARD
        )

    if hspace is None:
        hspace = (
            FIGURE_HSPACE_MULTIROW
            if nrows > 1
            else 0.08
        )

    figure.subplots_adjust(
        top=top,
        bottom=bottom,
        left=left,
        right=right,
        wspace=wspace,
        hspace=hspace,
    )


def set_axis_bounds(
    axis,
    bounds,
):
    if bounds is None:
        return

    xmin, ymin, xmax, ymax = (
        bounds
    )

    axis.set_xlim(
        xmin,
        xmax,
    )
    axis.set_ylim(
        ymin,
        ymax,
    )
    axis.set_aspect(
        "equal",
        adjustable="box",
    )


def style_canary_inset(
    inset_axis,
    subtitle=None,
):
    set_axis_bounds(
        inset_axis,
        CANARY_BOUNDS,
    )

    inset_axis.set_xticks(
        []
    )
    inset_axis.set_yticks(
        []
    )
    inset_axis.set_facecolor(
        "white"
    )

    for spine in inset_axis.spines.values():
        spine.set_visible(
            True
        )
        spine.set_linewidth(
            INSET_BORDER_WIDTH
        )
        spine.set_edgecolor(
            INSET_BORDER_COLOUR
        )

    inset_axis.text(
        0.03,
        0.96,
        CANARY_INSET_LABEL,
        transform=inset_axis.transAxes,
        ha="left",
        va="top",
        fontsize=INSET_LABEL_FONTSIZE,
        fontweight="bold",
        zorder=20,
    )

    if (
        CANARY_INSET_SHOW_SUBTITLE
        and subtitle
    ):
        inset_axis.text(
            0.03,
            0.85,
            subtitle,
            transform=inset_axis.transAxes,
            ha="left",
            va="top",
            fontsize=max(
                INSET_LABEL_FONTSIZE
                - 1,
                6,
            ),
            color="#555555",
            zorder=20,
        )

    if INSET_SCALE_BAR:
        add_scale_bar(
            inset_axis,
            location=(
                0.07,
                0.07,
            ),
            linewidth=1.1,
        )


def create_canary_inset(
    parent_axis,
    subtitle=None,
):
    if not ENABLE_CANARY_INSET:
        return None

    inset_axis = parent_axis.inset_axes(
        RESOLVED_CANARY_INSET_BOUNDS
    )

    style_canary_inset(
        inset_axis,
        subtitle=subtitle,
    )

    return inset_axis


def draw_polygon_background(
    axis,
    polygon_frame,
    province_frame=None,
):
    polygon_frame.plot(
        ax=axis,
        facecolor=DISTRICT_FACE,
        edgecolor=DISTRICT_EDGE,
        linewidth=BACKGROUND_EDGEWIDTH,
        zorder=1,
    )

    if (
        province_frame
        is not None
        and not province_frame.empty
    ):
        province_frame.boundary.plot(
            ax=axis,
            color=PROVINCE_EDGE,
            linewidth=PROVINCE_EDGEWIDTH,
            zorder=5,
        )


def draw_background(
    axis,
    province_overlay=True,
    inset_subtitle=None,
):
    draw_polygon_background(
        axis,
        main_map_districts,
        (
            main_map_provinces
            if province_overlay
            else None
        ),
    )

    set_axis_bounds(
        axis,
        MAIN_MAP_BOUNDS,
    )

    inset_axis = create_canary_inset(
        axis,
        subtitle=inset_subtitle,
    )

    if inset_axis is not None:
        draw_polygon_background(
            inset_axis,
            canary_districts,
            (
                canary_provinces
                if province_overlay
                else None
            ),
        )

        set_axis_bounds(
            inset_axis,
            CANARY_BOUNDS,
        )

    return inset_axis


def line_segments_from_frame(
    frame,
):
    if frame.empty:
        return np.empty(
            (
                0,
                2,
                2,
            ),
            dtype=float,
        )

    origins = frame[
        [
            "origin_x",
            "origin_y",
        ]
    ].to_numpy(
        dtype=float
    )

    destinations = frame[
        [
            "destination_x",
            "destination_y",
        ]
    ].to_numpy(
        dtype=float
    )

    return np.stack(
        [
            origins,
            destinations,
        ],
        axis=1,
    )


def add_line_collection(
    axis,
    frame,
    colour,
    linewidths,
    alpha,
    zorder,
):
    if frame.empty:
        return None

    segments = line_segments_from_frame(
        frame
    )

    collection = LineCollection(
        segments,
        colors=colour,
        linewidths=linewidths,
        alpha=alpha,
        zorder=zorder,
        rasterized=FLOW_RASTERIZED,
    )

    axis.add_collection(
        collection
    )

    return collection


def render_flow_region(
    axis,
    frame,
    weight_column,
    colour,
    scale,
    render_mode=FLOW_RENDER_MODE,
    all_alpha=ALL_EDGE_ALPHA,
    backbone_alpha=BACKBONE_ALPHA,
):
    if axis is None or frame.empty:
        return {
            "displayed_edges": 0,
            "backbone_edges": 0,
        }

    if render_mode == "all_plus_backbone":
        add_line_collection(
            axis,
            frame,
            colour,
            linewidths=np.full(
                len(frame),
                ALL_EDGE_WIDTH,
                dtype=float,
            ),
            alpha=all_alpha,
            zorder=2,
        )

        backbone = frame.loc[
            frame[weight_column]
            >= scale[
                "backbone_threshold"
            ]
        ].copy()

        if not backbone.empty:
            add_line_collection(
                axis,
                backbone,
                colour,
                linewidths=scaled_linewidth(
                    backbone[
                        weight_column
                    ],
                    scale=scale,
                ),
                alpha=backbone_alpha,
                zorder=4,
            )

        return {
            "displayed_edges": int(
                len(frame)
            ),
            "backbone_edges": int(
                len(backbone)
            ),
        }

    if render_mode == "weighted_all":
        add_line_collection(
            axis,
            frame,
            colour,
            linewidths=scaled_linewidth(
                frame[
                    weight_column
                ],
                scale=scale,
            ),
            alpha=WEIGHTED_ALL_ALPHA,
            zorder=3,
        )

        return {
            "displayed_edges": int(
                len(frame)
            ),
            "backbone_edges": int(
                len(frame)
            ),
        }

    if render_mode == "uniform_all":
        add_line_collection(
            axis,
            frame,
            colour,
            linewidths=np.full(
                len(frame),
                ALL_EDGE_WIDTH,
                dtype=float,
            ),
            alpha=all_alpha,
            zorder=3,
        )

        return {
            "displayed_edges": int(
                len(frame)
            ),
            "backbone_edges": 0,
        }

    raise ValueError(
        "Unknown FLOW_RENDER_MODE. "
        "Use all_plus_backbone, "
        "weighted_all or uniform_all."
    )


def render_flow_layer(
    main_axis,
    inset_axis,
    frame,
    weight_column,
    colour,
    scale,
    render_mode=FLOW_RENDER_MODE,
    all_alpha=ALL_EDGE_ALPHA,
    backbone_alpha=BACKBONE_ALPHA,
):
    main_part = frame.loc[
        frame["_edge_map_region"]
        == "main"
    ].copy()

    canary_part = frame.loc[
        frame["_edge_map_region"]
        == "canary"
    ].copy()

    main_diagnostics = render_flow_region(
        main_axis,
        main_part,
        weight_column,
        colour,
        scale,
        render_mode=render_mode,
        all_alpha=all_alpha,
        backbone_alpha=backbone_alpha,
    )

    canary_diagnostics = render_flow_region(
        inset_axis,
        canary_part,
        weight_column,
        colour,
        scale,
        render_mode=render_mode,
        all_alpha=all_alpha,
        backbone_alpha=backbone_alpha,
    )

    return {
        "main_displayed_edges": (
            main_diagnostics[
                "displayed_edges"
            ]
        ),
        "canary_displayed_edges": (
            canary_diagnostics[
                "displayed_edges"
            ]
        ),
        "main_backbone_edges": (
            main_diagnostics[
                "backbone_edges"
            ]
        ),
        "canary_backbone_edges": (
            canary_diagnostics[
                "backbone_edges"
            ]
        ),
    }


def plot_flow_network(
    axis,
    frame,
    weight_column,
    colour,
    title,
    scale,
    top_n=NATIONAL_TOP_N,
    selection_mode=FLOW_FILTER_MODE,
    render_mode=FLOW_RENDER_MODE,
    province_overlay=True,
    all_alpha=ALL_EDGE_ALPHA,
    backbone_alpha=BACKBONE_ALPHA,
):
    inset_axis = draw_background(
        axis,
        province_overlay=province_overlay,
        inset_subtitle=CANARY_INSET_SUBTITLE,
    )

    main_candidates = frame.loc[
        frame["_edge_map_region"]
        == "main"
    ].copy()

    canary_candidates = frame.loc[
        frame["_edge_map_region"]
        == "canary"
    ].copy()

    cross_candidates = frame.loc[
        frame["_edge_map_region"]
        == "cross_window"
    ].copy()

    main_mapped = filter_edges(
        main_candidates,
        weight_column,
        mode=selection_mode,
        top_n=top_n,
    )

    canary_mapped = filter_edges(
        canary_candidates,
        weight_column,
        mode=selection_mode,
        top_n=top_n,
    )

    selected = pd.concat(
        [
            main_mapped,
            canary_mapped,
        ],
        ignore_index=False,
    )

    diagnostics = render_flow_layer(
        axis,
        inset_axis,
        selected,
        weight_column,
        colour,
        scale,
        render_mode=render_mode,
        all_alpha=all_alpha,
        backbone_alpha=backbone_alpha,
    )

    cross_positive = _positive_external_edges(
        cross_candidates,
        weight_column,
    )

    all_external = _positive_external_edges(
        frame,
        weight_column,
    )

    selected.attrs[
        "filter_mode"
    ] = selection_mode

    selected.attrs[
        "render_mode"
    ] = render_mode

    selected.attrs[
        "candidate_external_edges"
    ] = int(
        len(all_external)
    )

    selected.attrs[
        "displayed_external_edges"
    ] = int(
        len(selected)
    )

    selected.attrs[
        "candidate_external_weight"
    ] = float(
        all_external[
            weight_column
        ].sum()
    )

    selected.attrs[
        "displayed_external_weight"
    ] = float(
        selected[
            weight_column
        ].sum()
    )

    selected.attrs[
        "cross_window_edges_omitted"
    ] = int(
        len(
            cross_positive
        )
    )

    selected.attrs[
        "cross_window_weight_omitted"
    ] = float(
        cross_positive[
            weight_column
        ].sum()
    )

    selected.attrs[
        "self_loops_excluded"
    ] = int(
        (
            (
                frame["origin"]
                == frame[
                    "destination"
                ]
            )
            & (
                pd.to_numeric(
                    frame[
                        weight_column
                    ],
                    errors="coerce",
                )
                > FLOW_MIN_WEIGHT
            )
        ).sum()
    )

    for key, value in diagnostics.items():
        selected.attrs[
            key
        ] = value

    finish_map_axis(
        axis,
        title,
        has_canary_inset=True,
    )

    return selected


def add_colourbar(
    figure,
    axis,
    cmap,
    norm,
    label,
):
    divider = make_axes_locatable(
        axis
    )

    colour_axis = divider.append_axes(
        "right",
        size="3.2%",
        pad=0.08,
    )

    colourbar = figure.colorbar(
        ScalarMappable(
            norm=norm,
            cmap=cmap,
        ),
        cax=colour_axis,
    )

    colourbar.set_label(
        label
    )
    colourbar.ax.yaxis.set_major_formatter(
        _plain_tick_formatter(
            decimals=COLOURBAR_MAX_DECIMALS,
        )
    )
    colourbar.ax.yaxis.get_offset_text().set_visible(False)
    colourbar.update_ticks()

    return colourbar



def plot_region_values(
    axis,
    inset_axis,
    geodata,
    column,
    cmap,
    norm,
    province_overlay=True,
    categorical=False,
    vmin=None,
    vmax=None,
):
    main_values = geodata.loc[
        ~geodata[
            "district_id"
        ].isin(
            CANARY_DISTRICT_IDS
        )
    ].copy()

    canary_values = geodata.loc[
        geodata[
            "district_id"
        ].isin(
            CANARY_DISTRICT_IDS
        )
    ].copy()

    plot_kwargs = {
        "column": column,
        "cmap": cmap,
        "edgecolor": DISTRICT_EDGE,
        "linewidth": BACKGROUND_EDGEWIDTH,
        "missing_kwds": {
            "color": NO_DATA_FACE,
            "edgecolor": DISTRICT_EDGE,
            "label": MISSING_LABEL,
        },
        "zorder": 2,
    }

    if categorical:
        plot_kwargs.update(
            {
                "vmin": vmin,
                "vmax": vmax,
            }
        )
    else:
        plot_kwargs.update(
            {
                "norm": norm,
            }
        )

    main_values.plot(
        ax=axis,
        **plot_kwargs,
    )

    set_axis_bounds(
        axis,
        MAIN_MAP_BOUNDS,
    )

    if (
        province_overlay
        and main_map_provinces
        is not None
    ):
        main_map_provinces.boundary.plot(
            ax=axis,
            color=PROVINCE_EDGE,
            linewidth=PROVINCE_EDGEWIDTH,
            zorder=4,
        )

    if inset_axis is not None:
        canary_values.plot(
            ax=inset_axis,
            **plot_kwargs,
        )

        if (
            province_overlay
            and canary_provinces
            is not None
        ):
            canary_provinces.boundary.plot(
                ax=inset_axis,
                color=PROVINCE_EDGE,
                linewidth=PROVINCE_EDGEWIDTH,
                zorder=4,
            )

        set_axis_bounds(
            inset_axis,
            CANARY_BOUNDS,
        )

def plot_continuous_choropleth(
    axis,
    figure,
    geodata,
    column,
    title,
    cmap=SEQUENTIAL_CMAP,
    diverging=False,
    label=None,
    shared_norm=None,
    province_overlay=True,
    add_colorbar=True,
):
    values = pd.to_numeric(
        geodata[column],
        errors="coerce",
    )

    if shared_norm is not None:
        norm = shared_norm

    elif diverging:
        norm = shared_diverging_norm(
            values
        )

    else:
        norm = shared_sequential_norm(
            values
        )

    inset_axis = create_canary_inset(
        axis
    )

    plot_region_values(
        axis,
        inset_axis,
        geodata,
        column,
        cmap,
        norm,
        province_overlay=province_overlay,
    )

    overlay_true_missing(
        axis,
        inset_axis,
        geodata,
        column,
    )

    if add_colorbar:
        add_colourbar(
            figure,
            axis,
            cmap,
            norm,
            label or column,
        )

    finish_map_axis(
        axis,
        title,
        has_canary_inset=True,
    )

    return norm


def plot_discrete_choropleth(
    axis,
    geodata,
    column,
    title,
    cmap,
    vmin,
    vmax,
    province_overlay=True,
    legend_handles=None,
    legend_location="lower right",
):
    inset_axis = create_canary_inset(
        axis
    )

    plot_region_values(
        axis,
        inset_axis,
        geodata,
        column,
        cmap,
        norm=None,
        province_overlay=province_overlay,
        categorical=True,
        vmin=vmin,
        vmax=vmax,
    )

    if legend_handles:
        axis.legend(
            handles=legend_handles,
            frameon=False,
            loc=legend_location,
        )

    finish_map_axis(
        axis,
        title,
        has_canary_inset=True,
    )

    return inset_axis


def spatial_join_district_values(frame=None, role=None, sample_only=False):
    working = period_premiums_display.copy() if frame is None or frame is period_premiums else frame.copy()
    if role is not None and "role" in working.columns:
        working = working.loc[working["role"] == role].copy()
    if sample_only and "period_analysis_sample" in working.columns:
        working = working.loc[working["period_analysis_sample"]].copy()
    working = working.drop_duplicates("district_id")
    mapped = districts.merge(working, on="district_id", how="left", validate="one_to_one")
    if "_display_status" not in mapped.columns:
        mapped["_display_status"] = "No valid analytical observation"
    return mapped


def overlay_true_missing(axis, inset_axis, geodata, value_column):
    if not SHOW_TRUE_MISSING_AS_HATCH:
        return
    missing = geodata.loc[pd.to_numeric(geodata[value_column], errors="coerce").isna()].copy()
    if missing.empty:
        return
    main_missing = missing.loc[~missing["district_id"].isin(CANARY_DISTRICT_IDS)]
    canary_missing = missing.loc[missing["district_id"].isin(CANARY_DISTRICT_IDS)]
    if not main_missing.empty:
        main_missing.plot(
            ax=axis, facecolor="none", edgecolor=TRUE_MISSING_EDGE,
            linewidth=0.18, hatch=TRUE_MISSING_HATCH, zorder=8,
        )
    if inset_axis is not None and not canary_missing.empty:
        canary_missing.plot(
            ax=inset_axis, facecolor="none", edgecolor=TRUE_MISSING_EDGE,
            linewidth=0.18, hatch=TRUE_MISSING_HATCH, zorder=8,
        )

# ---------------------------------------------------------------------
# Mapping-helper integrity audit
# ---------------------------------------------------------------------
_REQUIRED_MAPPING_HELPERS = [
    "plot_region_values",
    "plot_continuous_choropleth",
    "plot_discrete_choropleth",
    "overlay_true_missing",
    "create_canary_inset",
    "add_colourbar",
    "plot_flow_network",
]

_missing_mapping_helpers = [
    name
    for name in _REQUIRED_MAPPING_HELPERS
    if name not in globals() or not callable(globals()[name])
]

if _missing_mapping_helpers:
    raise RuntimeError(
        "The mapping helper cell is incomplete. Missing callable helpers: "
        f"{_missing_mapping_helpers}"
    )

print(
    "Mapping-helper audit passed:",
    ", ".join(_REQUIRED_MAPPING_HELPERS),
)



## 6A. Geography and analytical coverage QA

In [ ]:
formal_polygon_coverage = (
    len(formal_district_ids & set(districts["district_id"]))
    / max(len(formal_district_ids), 1)
)
if formal_polygon_coverage < MIN_FORMAL_ID_POLYGON_COVERAGE:
    message = (
        f"Formal district polygon coverage is {formal_polygon_coverage:.3%}, "
        f"below {MIN_FORMAL_ID_POLYGON_COVERAGE:.3%}."
    )
    if FAIL_ON_COVERAGE_ERROR:
        raise RuntimeError(message)
    warnings.warn(message)

geography_qa = pd.DataFrame([{
    "formal_district_ids": len(formal_district_ids),
    "mapped_district_polygons": districts["district_id"].nunique(),
    "formal_polygon_coverage": formal_polygon_coverage,
    "canary_districts": len(canary_districts),
    "main_map_districts": len(main_map_districts),
}])
atomic_to_csv(geography_qa, LOG_DIR / "geography_analytical_qa.csv")
display(geography_qa)


# Part I. National networks across hybrid-work intensity

The main map compares three representative layers. A six-panel appendix map retains the full intensity gradient. All panels use the same normalised flow measure and one common line-width scale.


## 10. Prepare period layer edge geometries and common flow scale


In [ ]:
period_layer_edge_gdf = edges_to_geodataframe(period_layer_edges)
layer_scale = build_flow_scale(
    period_layer_edge_gdf["weight_share"],
    label="Share of layer flow (%)",
    measure_type="share",
    scale_name="six_layer_network_share",
)
print("Mapped period layer edges:", len(period_layer_edge_gdf))


## 11. All hybrid-work intensity layers

All six layers are shown in one directly comparable main figure. The panels share the same edge selection, line-width scale, map extent and flow legend.


In [ ]:
# Remove obsolete representative-layer outputs so that stale files do not
# remain in the publication folders.
if RESET_LAYER_MAP_OUTPUTS:
    obsolete_stems = [
        "figure_1_representative_network_intensity_layers",
        "appendix_figure_all_six_network_intensity_layers",
        "figure_1_job_home_networks_across_all_intensity_layers",
    ]
    for directory in [FIGURE_MAIN_DIR, FIGURE_APPENDIX_DIR]:
        for stem in obsolete_stems:
            for suffix in [".png", ".pdf", ".svg"]:
                path = directory / f"{stem}{suffix}"
                if path.exists():
                    path.unlink()

figure, axes = plt.subplots(
    3,
    2,
    figsize=ALL_LAYER_FIGSIZE,
)

apply_map_layout(
    figure,
    nrows=3,
    right=ALL_LAYER_RIGHT,
    top=ALL_LAYER_TOP,
    bottom=ALL_LAYER_BOTTOM,
    wspace=ALL_LAYER_WSPACE,
    hspace=ALL_LAYER_HSPACE,
)

all_layer_map_diagnostics = []

for panel_index, (axis, band) in enumerate(
    zip(axes.flat, MAIN_NETWORK_MAP_BANDS)
):
    frame = period_layer_edge_gdf.loc[
        period_layer_edge_gdf["recurrence_band"] == band
    ].copy()

    intensity = RECURRENCE_INTENSITY[band]
    title = (
        f"{BAND_DISPLAY_LABELS[band]} workplace days"
        f"\nHybrid-work intensity = {intensity:.2f}"
    )

    selected = plot_flow_network(
        axis,
        frame,
        "weight_share",
        LAYER_COLOURS[band],
        title,
        scale=layer_scale,
        top_n=PANEL_TOP_N,
    )

    all_layer_map_diagnostics.append({
        "panel": panel_index + 1,
        "recurrence_band": band,
        "workplace_days_label": BAND_DISPLAY_LABELS[band],
        "hybrid_work_intensity": intensity,
        "candidate_external_edges": selected.attrs.get(
            "candidate_external_edges"
        ),
        "displayed_external_edges": selected.attrs.get(
            "displayed_external_edges"
        ),
        "candidate_external_weight": selected.attrs.get(
            "candidate_external_weight"
        ),
        "displayed_external_weight": selected.attrs.get(
            "displayed_external_weight"
        ),
        "cross_window_edges_omitted": selected.attrs.get(
            "cross_window_edges_omitted"
        ),
    })

add_figure_flow_legend(
    figure,
    layer_scale,
    bbox_to_anchor=ALL_LAYER_LEGEND_ANCHOR,
    colour="#555555",
    title="Share of layer flow",
)

figure.suptitle(
    "Job–home networks across all hybrid-work intensity levels",
    y=0.980,
)

save_figure(
    figure,
    "figure_1_job_home_networks_across_all_intensity_layers",
    directory=FIGURE_MAIN_DIR,
)

all_layer_map_diagnostics = pd.DataFrame(all_layer_map_diagnostics)
atomic_to_csv(
    all_layer_map_diagnostics,
    LOG_DIR / "all_intensity_layer_map_diagnostics.csv",
)
display(all_layer_map_diagnostics)


## 12. Individual hybrid-work intensity layer maps (appendix)

Each of the six layers is also exported separately at publication resolution. All individual maps use the same national flow scale as the combined figure, so line widths remain comparable.


In [ ]:
individual_layer_inventory = []

if EXPORT_INDIVIDUAL_LAYER_MAPS:
    for band in RECURRENCE_BAND_ORDER:
        frame = period_layer_edge_gdf.loc[
            period_layer_edge_gdf["recurrence_band"] == band
        ].copy()

        intensity = RECURRENCE_INTENSITY[band]
        figure, axis = plt.subplots(
            1,
            1,
            figsize=INDIVIDUAL_LAYER_FIGSIZE,
        )

        apply_map_layout(
            figure,
            nrows=1,
            right=INDIVIDUAL_LAYER_RIGHT,
            top=0.88,
            bottom=0.075,
            left=0.025,
            wspace=0.03,
        )

        selected = plot_flow_network(
            axis,
            frame,
            "weight_share",
            LAYER_COLOURS[band],
            (
                f"{BAND_DISPLAY_LABELS[band]} workplace days "
                f"(hybrid-work intensity = {intensity:.2f})"
            ),
            scale=layer_scale,
            top_n=PANEL_TOP_N,
        )

        add_figure_flow_legend(
            figure,
            layer_scale,
            bbox_to_anchor=INDIVIDUAL_LAYER_LEGEND_ANCHOR,
            colour="#555555",
            title="Share of layer flow",
        )

        figure.suptitle(
            "Job–home network by hybrid-work intensity",
            y=0.965,
        )

        stem = (
            "appendix_figure_network_intensity_layer_"
            f"{LAYER_FILE_LABELS[band]}"
        )
        saved_paths = save_figure(
            figure,
            stem,
            directory=FIGURE_APPENDIX_DIR,
        )

        individual_layer_inventory.append({
            "recurrence_band": band,
            "workplace_days_label": BAND_DISPLAY_LABELS[band],
            "hybrid_work_intensity": intensity,
            "figure_stem": stem,
            "saved_files": "|".join(str(path) for path in saved_paths),
            "candidate_external_edges": selected.attrs.get(
                "candidate_external_edges"
            ),
            "displayed_external_edges": selected.attrs.get(
                "displayed_external_edges"
            ),
        })

individual_layer_inventory = pd.DataFrame(individual_layer_inventory)
atomic_to_csv(
    individual_layer_inventory,
    LOG_DIR / "individual_intensity_layer_figure_inventory.csv",
)
display(individual_layer_inventory)

if EXPORT_INDIVIDUAL_LAYER_MAPS:
    exported_bands = set(
        individual_layer_inventory["recurrence_band"].astype(str)
    )
    missing_bands = [
        band for band in RECURRENCE_BAND_ORDER
        if band not in exported_bands
    ]
    if missing_bands:
        raise RuntimeError(
            "Individual intensity-layer maps were not generated for: "
            f"{missing_bands}"
        )


# Part II. District hybrid-work intensity and network outcomes


## 13. District hybrid-work intensity


In [ ]:
intensity_maps = {}

for role in ["residential", "employment"]:
    role_values = district_period_display.loc[
        district_period_display["role"].eq(role),
        [
            "district_id",
            "hybrid_intensity_all_flows",
        ],
    ]

    intensity_maps[role] = districts.merge(
        role_values,
        on="district_id",
        how="left",
        validate="one_to_one",
    )

shared_intensity_norm = shared_sequential_norm(
    pd.concat(
        [
            intensity_maps["residential"][
                "hybrid_intensity_all_flows"
            ],
            intensity_maps["employment"][
                "hybrid_intensity_all_flows"
            ],
        ],
        ignore_index=True,
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(15.8, 7.6),
)

apply_map_layout(
    figure,
    nrows=1,
    right=FIGURE_RIGHT_WITH_LEGEND,
    top=0.86,
    bottom=0.09,
    wspace=0.05,
)

for axis, role in zip(
    axes,
    ["residential", "employment"],
):
    plot_continuous_choropleth(
        axis,
        figure,
        intensity_maps[role],
        "hybrid_intensity_all_flows",
        ROLE_LABELS.get(
            role,
            role.title(),
        ),
        cmap="viridis",
        shared_norm=shared_intensity_norm,
        label="Hybrid-work intensity",
        add_colorbar=False,
    )

add_colourbar(
    figure,
    axes[-1],
    "viridis",
    shared_intensity_norm,
    label="Hybrid-work intensity",
)

figure.suptitle(
    "District hybrid-work intensity in residential and employment roles"
)

save_figure(
    figure,
    "figure_2_district_hybrid_work_intensity",
    directory=FIGURE_MAIN_DIR,
)


## 14. District network outcomes


In [ ]:
OUTCOME_MAPS = {
    "partner_coverage": "Partner coverage",
    "partner_diversity": "Partner diversity",
    "mean_external_distance_km": "Mean external distance (km)",
}
DISTRICT_ROLE_ORDER = ["residential", "employment"]

figure, axes = plt.subplots(
    3,
    2,
    figsize=SIX_PANEL_FIGSIZE,
)

apply_map_layout(
    figure,
    nrows=3,
    right=SIX_PANEL_RIGHT,
    top=SIX_PANEL_TOP,
    bottom=SIX_PANEL_BOTTOM,
    wspace=SIX_PANEL_WSPACE,
    hspace=SIX_PANEL_HSPACE,
)

for outcome_index, (column, label) in enumerate(
    OUTCOME_MAPS.items()
):
    values = pd.concat(
        [
            district_period_display.loc[
                district_period_display["role"].eq(role),
                column,
            ]
            for role in DISTRICT_ROLE_ORDER
        ],
        ignore_index=True,
    )

    norm = shared_sequential_norm(values)

    for role_index, role in enumerate(DISTRICT_ROLE_ORDER):
        role_values = district_period_display.loc[
            district_period_display["role"].eq(role),
            ["district_id", column],
        ]

        mapped = districts.merge(
            role_values,
            on="district_id",
            how="left",
            validate="one_to_one",
        )

        plot_continuous_choropleth(
            axes[outcome_index, role_index],
            figure,
            mapped,
            column,
            (
                f"{ROLE_LABELS.get(role, role.title())}: "
                f"{label}"
            ),
            cmap="viridis",
            shared_norm=norm,
            label=label,
            add_colorbar=True,
        )

figure.suptitle(
    "District job–home network outcomes"
)

save_figure(
    figure,
    "figure_3_district_network_outcomes",
    directory=FIGURE_MAIN_DIR,
)


# Part III. Settlement context and conditional intensity relationships


## 15. Settlement urbanity and existing onsite network position


In [ ]:
# Settlement urbanity is a static district attribute and is therefore
# shown once. Existing onsite network position remains role-specific.

urbanity_values = (
    district_period_display[
        ["district_id", "settlement_urbanity"]
    ]
    .drop_duplicates("district_id")
)

urbanity_map = districts.merge(
    urbanity_values,
    on="district_id",
    how="left",
    validate="one_to_one",
)

position_maps = {}

for role in ["residential", "employment"]:
    role_values = district_period_display.loc[
        district_period_display["role"].eq(role),
        [
            "district_id",
            "onsite_network_position",
        ],
    ]

    position_maps[role] = districts.merge(
        role_values,
        on="district_id",
        how="left",
        validate="one_to_one",
    )

urbanity_norm = shared_sequential_norm(
    urbanity_map["settlement_urbanity"]
)

position_norm = shared_sequential_norm(
    pd.concat(
        [
            position_maps["residential"][
                "onsite_network_position"
            ],
            position_maps["employment"][
                "onsite_network_position"
            ],
        ],
        ignore_index=True,
    )
)

figure, axes = plt.subplots(
    1,
    3,
    figsize=(20.0, 7.2),
)

apply_map_layout(
    figure,
    nrows=1,
    right=0.965,
    top=0.86,
    bottom=0.09,
    wspace=0.11,
)

plot_continuous_choropleth(
    axes[0],
    figure,
    urbanity_map,
    "settlement_urbanity",
    "Settlement urbanity",
    cmap=URBANITY_CMAP,
    shared_norm=urbanity_norm,
    label="Settlement urbanity",
    add_colorbar=True,
)

for axis, role in zip(
    axes[1:],
    ["residential", "employment"],
):
    plot_continuous_choropleth(
        axis,
        figure,
        position_maps[role],
        "onsite_network_position",
        (
            f"{ROLE_LABELS.get(role, role.title())}: "
            "Existing onsite network position"
        ),
        cmap=HIERARCHY_CMAP,
        shared_norm=position_norm,
        label="Existing onsite network position",
        add_colorbar=True,
    )

figure.suptitle(
    "Settlement urbanity and existing onsite network position"
)

save_figure(
    figure,
    "figure_4_settlement_context_and_onsite_network_position",
    directory=FIGURE_MAIN_DIR,
)


## 16. Spatial variation in the marginal relationship with hybrid-work intensity (appendix)


In [ ]:
# Notebook 01 uses log1p_effective_partner_diversity as the canonical
# context-model outcome. Earlier mapping notebooks expected the shorter
# log1p_partner_diversity alias, which caused every valid diversity estimate
# to be rendered as missing. Each panel now resolves its source field
# explicitly and stops with an informative error if no compatible field is
# available.
MARGINAL_OUTCOME_SPECS = [
    {
        "canonical": "log1p_partner_coverage",
        "aliases": [
            "log1p_partner_coverage",
            "partner_coverage",
        ],
        "label": "Partner coverage",
    },
    {
        "canonical": "log1p_effective_partner_diversity",
        "aliases": [
            "log1p_effective_partner_diversity",
            "log1p_partner_diversity",
            "effective_partner_diversity",
            "partner_diversity",
        ],
        "label": "Partner diversity",
    },
    {
        "canonical": "log_mean_external_distance_km",
        "aliases": [
            "log_mean_external_distance_km",
            "mean_external_distance_km",
        ],
        "label": "Mean external distance",
    },
]
MARGINAL_ROLE_ORDER = ["residential", "employment"]

marginal_effects = marginal_effects.copy()
marginal_effects["_outcome_key"] = (
    marginal_effects["outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

available_marginal_outcomes = set(
    marginal_effects["_outcome_key"].dropna()
)


def resolve_marginal_outcome(specification):
    matching_aliases = [
        alias.lower()
        for alias in specification["aliases"]
        if alias.lower() in available_marginal_outcomes
    ]

    if not matching_aliases:
        raise KeyError(
            "No compatible marginal-effect outcome was found for "
            f"{specification['label']!r}. Expected one of "
            f"{specification['aliases']}; available outcomes are "
            f"{sorted(available_marginal_outcomes)}."
        )

    return matching_aliases[0]


resolved_marginal_specs = []
marginal_panel_data = {}
marginal_effect_parts = []
marginal_alias_audit_rows = []

for specification in MARGINAL_OUTCOME_SPECS:
    resolved_key = resolve_marginal_outcome(specification)
    resolved = {
        **specification,
        "resolved_key": resolved_key,
    }
    resolved_marginal_specs.append(resolved)

    for role in MARGINAL_ROLE_ORDER:
        values = marginal_effects.loc[
            marginal_effects["role"].eq(role)
            & marginal_effects["_outcome_key"].eq(resolved_key),
            [
                "district_id",
                "intensity_effect_per_0_1_pct",
            ],
        ].copy()

        duplicate_ids = values["district_id"].duplicated(
            keep=False
        )
        if duplicate_ids.any():
            examples = (
                values.loc[duplicate_ids, "district_id"]
                .astype(str)
                .drop_duplicates()
                .head(10)
                .tolist()
            )
            raise RuntimeError(
                "Marginal-effect input contains duplicate district rows "
                f"for {role}, {specification['label']}: {examples}"
            )

        valid_count = int(
            values["intensity_effect_per_0_1_pct"]
            .notna()
            .sum()
        )
        if valid_count == 0:
            raise RuntimeError(
                "No valid marginal effects are available for "
                f"{role}, {specification['label']}. "
                "Check the Notebook-01 context-model export."
            )

        marginal_panel_data[
            (role, specification["canonical"])
        ] = values
        marginal_effect_parts.append(
            values["intensity_effect_per_0_1_pct"]
        )
        marginal_alias_audit_rows.append({
            "role": role,
            "display_label": specification["label"],
            "canonical_outcome": specification["canonical"],
            "resolved_input_outcome": resolved_key,
            "input_rows": int(len(values)),
            "valid_effects": valid_count,
            "unique_districts": int(
                values["district_id"].nunique()
            ),
        })

marginal_outcome_alias_audit = pd.DataFrame(
    marginal_alias_audit_rows
)
atomic_to_csv(
    marginal_outcome_alias_audit,
    LOG_DIR / "marginal_outcome_alias_and_coverage_audit.csv",
)
display(marginal_outcome_alias_audit)

shared_effect_norm = shared_diverging_norm(
    pd.concat(
        marginal_effect_parts,
        ignore_index=True,
    )
)

figure, axes = plt.subplots(
    3,
    2,
    figsize=SIX_PANEL_FIGSIZE,
)
apply_map_layout(
    figure,
    nrows=3,
    right=SIX_PANEL_RIGHT,
    top=SIX_PANEL_TOP,
    bottom=SIX_PANEL_BOTTOM,
    wspace=SIX_PANEL_WSPACE,
    hspace=SIX_PANEL_HSPACE,
)

for outcome_index, specification in enumerate(
    resolved_marginal_specs
):
    for role_index, role in enumerate(MARGINAL_ROLE_ORDER):
        values = marginal_panel_data[
            (role, specification["canonical"])
        ]
        mapped = districts.merge(
            values,
            on="district_id",
            how="left",
            validate="one_to_one",
        )

        plot_continuous_choropleth(
            axes[outcome_index, role_index],
            figure,
            mapped,
            "intensity_effect_per_0_1_pct",
            (
                f"{ROLE_LABELS.get(role, role.title())}: "
                f"{specification['label']}"
            ),
            cmap=DIVERGING_CMAP,
            diverging=True,
            shared_norm=shared_effect_norm,
            label=(
                "Associated change for a 0.10 increase "
                "in intensity (%)"
            ),
            add_colorbar=False,
        )

# A figure-level colour bar preserves equal panel widths.
colour_axis = figure.add_axes(
    [0.900, 0.175, 0.014, 0.650]
)
colourbar = figure.colorbar(
    ScalarMappable(
        norm=shared_effect_norm,
        cmap=DIVERGING_CMAP,
    ),
    cax=colour_axis,
)
colourbar.set_label(
    "Associated change for a 0.10 increase in intensity (%)"
)
colourbar.ax.yaxis.set_major_formatter(
    _plain_tick_formatter(
        decimals=COLOURBAR_MAX_DECIMALS,
    )
)
colourbar.ax.yaxis.get_offset_text().set_visible(False)
colourbar.update_ticks()

figure.suptitle(
    "Conditional district relationships between hybrid-work "
    "intensity and network outcomes"
)
save_figure(
    figure,
    "appendix_figure_spatial_marginal_intensity_relationships",
    directory=FIGURE_APPENDIX_DIR,
)


## 17. Bivariate appendix maps

The bivariate appendix restores the 3 × 3 legend structure and colour scheme used in the earlier BIVARIATE_APPENDIX module. Class breaks are calculated globally across both roles for each variable pair, allowing direct residential–employment comparison.


In [ ]:
def quantile_breaks(
    values,
    quantiles=BIVARIATE_QUANTILES,
):
    series = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).dropna()

    if series.empty:
        return (0.0, 1.0)

    breaks = series.quantile(
        list(quantiles)
    ).to_numpy(dtype=float)

    if len(np.unique(breaks)) < len(breaks):
        unique_values = np.sort(
            series.unique()
        )

        if len(unique_values) >= 3:
            breaks = np.quantile(
                unique_values,
                list(quantiles),
            )
        elif len(unique_values) == 2:
            breaks = np.asarray(
                [
                    unique_values[0],
                    unique_values[1],
                ],
                dtype=float,
            )
        else:
            breaks = np.asarray(
                [
                    unique_values[0],
                    unique_values[0],
                ],
                dtype=float,
            )

    return tuple(
        float(value)
        for value in breaks
    )


def class_from_breaks(
    series,
    breaks,
):
    values = pd.to_numeric(
        series,
        errors="coerce",
    )

    result = pd.Series(
        np.nan,
        index=series.index,
    )

    valid = values.notna()

    result.loc[valid] = np.select(
        [
            values.loc[valid] <= breaks[0],
            values.loc[valid] <= breaks[1],
        ],
        [0, 1],
        default=2,
    )

    return result


def add_bivariate_legend(
    axis,
    x_label,
    y_label,
    x_categories=("Lower", "Medium", "Higher"),
    y_categories=("Lower", "Medium", "Higher"),
):
    legend_axis = axis.inset_axes(
        BIVARIATE_LEGEND_POSITION
    )

    for row in range(3):
        for column in range(3):
            colour_index = row * 3 + column

            legend_axis.add_patch(
                mpl.patches.Rectangle(
                    (column, row),
                    1,
                    1,
                    facecolor=BIVARIATE_COLOURS[
                        colour_index
                    ],
                    edgecolor="white",
                )
            )

    legend_axis.set_xlim(0, 3)
    legend_axis.set_ylim(0, 3)

    legend_axis.set_xticks(
        [0.5, 1.5, 2.5],
        x_categories,
        fontsize=6.5,
    )
    legend_axis.set_yticks(
        [0.5, 1.5, 2.5],
        y_categories,
        fontsize=6.5,
    )

    legend_axis.set_xlabel(
        x_label,
        fontsize=8,
        labelpad=6,
    )
    legend_axis.set_ylabel(
        y_label,
        fontsize=8,
        labelpad=8,
    )

    legend_axis.tick_params(
        length=0,
    )
    legend_axis.tick_params(
        axis="x",
        pad=2,
    )
    legend_axis.tick_params(
        axis="y",
        pad=2,
    )

    for spine in legend_axis.spines.values():
        spine.set_visible(False)

    return legend_axis


def bivariate_map(
    frame,
    role,
    x_column,
    y_column,
    x_breaks,
    y_breaks,
    title,
    stem,
    x_label,
    y_label,
):
    role_values = frame.loc[
        frame["role"].eq(role),
        [
            "district_id",
            x_column,
            y_column,
        ],
    ].copy()

    mapped = districts.merge(
        role_values,
        on="district_id",
        how="left",
        validate="one_to_one",
    )

    mapped["x_class"] = class_from_breaks(
        mapped[x_column],
        x_breaks,
    )
    mapped["y_class"] = class_from_breaks(
        mapped[y_column],
        y_breaks,
    )

    mapped["bivariate_class"] = (
        mapped["y_class"] * 3
        + mapped["x_class"]
    )

    bivariate_cmap = ListedColormap(
        BIVARIATE_COLOURS
    )

    figure, axis = plt.subplots(
        figsize=(9.6, 8.4)
    )

    apply_map_layout(
        figure,
        nrows=1,
        top=0.90,
        bottom=0.10,
    )

    inset_axis = create_canary_inset(
        axis
    )

    plot_region_values(
        axis,
        inset_axis,
        mapped,
        "bivariate_class",
        bivariate_cmap,
        norm=None,
        province_overlay=True,
        categorical=True,
        vmin=0,
        vmax=8,
    )

    overlay_true_missing(
        axis,
        inset_axis,
        mapped,
        "bivariate_class",
    )

    add_bivariate_legend(
        axis,
        x_label=x_label,
        y_label=y_label,
    )

    finish_map_axis(
        axis,
        title,
        has_canary_inset=True,
    )

    save_figure(
        figure,
        stem,
        directory=FIGURE_APPENDIX_DIR,
    )

    return mapped


bivariate_break_rows = []
bivariate_inventory_rows = []

if RUN_BIVARIATE_APPENDIX:
    bivariate_specs = [
        {
            "x": "hybrid_intensity_all_flows",
            "x_label": "Hybrid-work intensity",
            "y": "partner_coverage",
            "y_label": "Partner coverage",
            "stem": "intensity_partner_coverage",
        },
        {
            "x": "hybrid_intensity_all_flows",
            "x_label": "Hybrid-work intensity",
            "y": "partner_diversity",
            "y_label": "Partner diversity",
            "stem": "intensity_partner_diversity",
        },
        {
            "x": "hybrid_intensity_all_flows",
            "x_label": "Hybrid-work intensity",
            "y": "mean_external_distance_km",
            "y_label": "Mean external distance",
            "stem": "intensity_mean_external_distance",
        },
        {
            "x": "hybrid_intensity_all_flows",
            "x_label": "Hybrid-work intensity",
            "y": "settlement_urbanity",
            "y_label": "Settlement urbanity",
            "stem": "intensity_settlement_urbanity",
        },
        {
            "x": "hybrid_intensity_all_flows",
            "x_label": "Hybrid-work intensity",
            "y": "onsite_network_position",
            "y_label": "Existing onsite position",
            "stem": "intensity_onsite_position",
        },
    ]

    for specification in bivariate_specs:
        x_column = specification["x"]
        y_column = specification["y"]

        x_breaks = quantile_breaks(
            district_period_display[x_column]
        )
        y_breaks = quantile_breaks(
            district_period_display[y_column]
        )

        for variable, breaks in [
            (x_column, x_breaks),
            (y_column, y_breaks),
        ]:
            bivariate_break_rows.append({
                "specification": specification["stem"],
                "variable": variable,
                "lower_break": breaks[0],
                "upper_break": breaks[1],
            })

        for role in ["residential", "employment"]:
            stem = (
                "appendix_figure_bivariate_"
                f"{specification['stem']}_{role}"
            )

            mapped = bivariate_map(
                district_period_display,
                role=role,
                x_column=x_column,
                y_column=y_column,
                x_breaks=x_breaks,
                y_breaks=y_breaks,
                title=(
                    f"{ROLE_LABELS.get(role, role.title())}: "
                    f"{specification['x_label']} and "
                    f"{specification['y_label']}"
                ),
                stem=stem,
                x_label=specification["x_label"],
                y_label=specification["y_label"],
            )

            bivariate_inventory_rows.append({
                "role": role,
                "specification": specification["stem"],
                "figure_stem": stem,
                "valid_districts": int(
                    mapped["bivariate_class"].notna().sum()
                ),
            })

bivariate_break_log = pd.DataFrame(
    bivariate_break_rows
)
bivariate_inventory = pd.DataFrame(
    bivariate_inventory_rows
)

atomic_to_csv(
    bivariate_break_log,
    LOG_DIR / "bivariate_common_class_breaks.csv",
)
atomic_to_csv(
    bivariate_inventory,
    LOG_DIR / "bivariate_figure_inventory.csv",
)

display(bivariate_break_log)
display(bivariate_inventory)


## 18. Export map-ready layers and reproducibility manifest


In [ ]:
atomic_to_parquet(
    period_layer_edge_gdf,
    DATA_EXPORT_DIR / "period_layer_network_edges_geometries.parquet",
)
atomic_to_parquet(
    district_period_display,
    DATA_EXPORT_DIR / "district_period_intensity_outcomes_display.parquet",
)
atomic_to_parquet(
    marginal_effects,
    DATA_EXPORT_DIR / "district_marginal_intensity_relationships.parquet",
)
districts.to_parquet(
    DATA_EXPORT_DIR / "spain_master_district_geography.parquet",
    index=False,
)

figure_inventory = []
for directory, category in [
    (FIGURE_MAIN_DIR, "main"),
    (FIGURE_APPENDIX_DIR, "appendix"),
]:
    for path in sorted(directory.glob("*")):
        if path.is_file():
            figure_inventory.append({
                "category": category,
                "filename": path.name,
                "path": portable_path_label(path),
                "size_bytes": path.stat().st_size,
            })
figure_inventory = pd.DataFrame(figure_inventory)
atomic_to_csv(figure_inventory, MAPPING_STAGE_DIR / "figure_inventory.csv")

input_paths = [
    PERIOD_LAYER_EDGE_FILE,
    DISTRICT_PERIOD_FILE,
    DISTRICT_MONTH_FILE,
    MARGINAL_EFFECT_FILE,
    CENTROID_LOOKUP_FILE,
    district_polygon_path,
]
signature = analysis_signature(input_paths)
manifest = {
    "run_version": MAPPING_RUN_VERSION,
    "run_date": MAPPING_RUN_DATE,
    "run_tag": MAPPING_RUN_TAG,
    "formal_run_tag": FORMAL_RUN_TAG,
    "marginal_effect_map_location": "appendix",
    "notebook": "02_Hybrid_Work_Job_Home_Networks_Systematic_Mapping_v4_2_20260724",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "mapping_signature": signature,
    "project_root": ".",
    "formal_analysis_stage": portable_path_label(FORMAL_STAGE_DIR),
    "mapping_stage": portable_path_label(MAPPING_STAGE_DIR),
    "district_polygon_path": portable_path_label(district_polygon_path),
    "map_crs": MAP_CRS,
    "recurrence_band_order": RECURRENCE_BAND_ORDER,
    "main_network_map_bands": MAIN_NETWORK_MAP_BANDS,
    "individual_layer_maps_exported": EXPORT_INDIVIDUAL_LAYER_MAPS,
    "source_analysis_notebook": formal_manifest.get("notebook"),
    "source_analysis_signature": formal_manifest.get("analysis_signature"),
    "ghs_profile_path": portable_path_label(
        formal_manifest.get("ghs_profile_path")
    ),
    "ghs_profile_origin": formal_manifest.get("ghs_profile_origin"),
    "ghsl_release": formal_manifest.get("ghsl_release"),
    "ghsl_epoch": formal_manifest.get("ghsl_epoch"),
    "ghsl_resolution_metres": formal_manifest.get("ghsl_resolution_metres"),
    "ghs_match_share": (
        float(ghs_match_share)
        if np.isfinite(ghs_match_share)
        else None
    ),
    "bivariate_appendix": RUN_BIVARIATE_APPENDIX,
    "layer_network_colour": LAYER_NETWORK_COLOUR,
    "district_period_file": portable_path_label(DISTRICT_PERIOD_FILE),
    "flow_legend_notation": "fixed_decimal",
    "flow_legend_max_decimals": FLOW_LEGEND_MAX_DECIMALS,
    "flow_filter_mode": FLOW_FILTER_MODE,
    "flow_render_mode": FLOW_RENDER_MODE,
    "canary_inset_enabled": ENABLE_CANARY_INSET,
    "canary_district_count": len(canary_districts),
    "main_map_district_count": len(main_map_districts),
}

expected_layer_stems = [
    "figure_1_job_home_networks_across_all_intensity_layers",
]
if EXPORT_INDIVIDUAL_LAYER_MAPS:
    expected_layer_stems.extend([
        (
            "appendix_figure_network_intensity_layer_"
            f"{LAYER_FILE_LABELS[band]}"
        )
        for band in RECURRENCE_BAND_ORDER
    ])

expected_extensions = []
if SAVE_PNG:
    expected_extensions.append(".png")
if SAVE_PDF:
    expected_extensions.append(".pdf")
if SAVE_SVG:
    expected_extensions.append(".svg")

missing_layer_outputs = []
for stem in expected_layer_stems:
    directory = (
        FIGURE_MAIN_DIR
        if stem.startswith("figure_1_")
        else FIGURE_APPENDIX_DIR
    )
    for suffix in expected_extensions:
        path = directory / f"{stem}{suffix}"
        if not path.exists():
            missing_layer_outputs.append(portable_path_label(path))

layer_output_audit = pd.DataFrame([{
    "expected_layer_figures": len(expected_layer_stems),
    "expected_file_formats": len(expected_extensions),
    "expected_files": len(expected_layer_stems) * len(expected_extensions),
    "missing_files": len(missing_layer_outputs),
    "all_six_layers_exported": (
        set(individual_layer_inventory["recurrence_band"].astype(str))
        == set(RECURRENCE_BAND_ORDER)
        if EXPORT_INDIVIDUAL_LAYER_MAPS
        else True
    ),
}])
atomic_to_csv(
    layer_output_audit,
    LOG_DIR / "layer_figure_output_audit.csv",
)
display(layer_output_audit)

if missing_layer_outputs:
    raise RuntimeError(
        "The following expected intensity-layer figure files are missing:\n"
        + "\n".join(missing_layer_outputs)
    )


atomic_write_json(manifest, LOG_DIR / "mapping_manifest.json")
print(json.dumps(manifest, indent=2))


In [ ]:
# ================================================================
# Collect all outputs from Notebook 01 and Notebook 02
# Edit EXPORT_ROOT only when a different destination is required.
# ================================================================

import os
from pathlib import Path
from datetime import datetime
from collections import Counter
import csv
import json
import shutil


# ----------------------------------------------------------------
# 1. User settings
# ----------------------------------------------------------------

EXPORT_ROOT = (
    PROJECT_ROOT
    / "07_collected_outputs"
    / "hybrid_work_outputs_v4_4_v4_2_20260724"
)

SOURCE_ANALYSIS_ROOT = PROJECT_ROOT / "05_analysis"

# When the stage variables are available, their exact directories are
# used. If this list remains empty, the most recent manifests are used.
SOURCE_STAGE_DIRS = []
for variable_name in [
    "FORMAL_STAGE_DIR",
    "MAPPING_STAGE_DIR",
]:
    stage_value = globals().get(variable_name)
    if stage_value is not None:
        SOURCE_STAGE_DIRS.append(
            Path(stage_value)
        )

CREATE_TIMESTAMPED_SUBFOLDER = False
OVERWRITE_EXISTING = True
INCLUDE_MAP_INPUTS = True
INCLUDE_LOGS = True


# ----------------------------------------------------------------
# 2. File-type and directory filters
# ----------------------------------------------------------------

FIGURE_SUFFIXES = {
    ".png",
    ".pdf",
    ".svg",
    ".jpg",
    ".jpeg",
}

RESULT_SUFFIXES = {
    ".csv",
    ".xlsx",
    ".xls",
    ".parquet",
    ".feather",
    ".json",
    ".geojson",
    ".gpkg",
    ".html",
    ".txt",
}

ALLOWED_SUFFIXES = (
    FIGURE_SUFFIXES
    | RESULT_SUFFIXES
)

SKIP_DIRECTORY_NAMES = {
    "cache",
    "caches",
    "checkpoint",
    "checkpoints",
    "tmp",
    "temp",
    "__pycache__",
    ".ipynb_checkpoints",
}

MAP_INPUT_DIRECTORY_NAMES = {
    "map_input",
    "map_inputs",
    "mapping_input",
    "mapping_inputs",
    "map_inputs_for_notebook_02",
}


# ----------------------------------------------------------------
# 3. Resolve source-stage directories
# ----------------------------------------------------------------

def latest_manifest(
    analysis_root,
    manifest_name,
):
    candidates = [
        path
        for path in analysis_root.rglob(manifest_name)
        if not any(
            part.lower() in SKIP_DIRECTORY_NAMES
            for part in path.parts
        )
    ]

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda path: path.stat().st_mtime,
    )


def stage_root_from_manifest(
    manifest_path,
):
    if manifest_path is None:
        return None

    if manifest_path.parent.name.lower() in {
        "log",
        "logs",
    }:
        return manifest_path.parent.parent

    return manifest_path.parent


if not SOURCE_STAGE_DIRS:
    analysis_manifest = latest_manifest(
        SOURCE_ANALYSIS_ROOT,
        "analysis_manifest.json",
    )
    mapping_manifest = latest_manifest(
        SOURCE_ANALYSIS_ROOT,
        "mapping_manifest.json",
    )

    SOURCE_STAGE_DIRS = [
        stage_root
        for stage_root in [
            stage_root_from_manifest(
                analysis_manifest
            ),
            stage_root_from_manifest(
                mapping_manifest
            ),
        ]
        if stage_root is not None
    ]

unique_stage_dirs = []
for stage_dir in SOURCE_STAGE_DIRS:
    resolved_stage_dir = Path(
        stage_dir
    ).resolve()

    if resolved_stage_dir not in unique_stage_dirs:
        unique_stage_dirs.append(
            resolved_stage_dir
        )

SOURCE_STAGE_DIRS = unique_stage_dirs

if not SOURCE_STAGE_DIRS:
    raise FileNotFoundError(
        "No Notebook-01 or Notebook-02 output directory was found. "
        "Check SOURCE_ANALYSIS_ROOT or set SOURCE_STAGE_DIRS manually."
    )

missing_stage_dirs = [
    str(path)
    for path in SOURCE_STAGE_DIRS
    if not path.exists()
]

if missing_stage_dirs:
    raise FileNotFoundError(
        "The following source output directories do not exist:\n"
        + "\n".join(missing_stage_dirs)
    )


# ----------------------------------------------------------------
# 4. Prepare the destination
# ----------------------------------------------------------------

if CREATE_TIMESTAMPED_SUBFOLDER:
    run_stamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    export_dir = (
        EXPORT_ROOT
        / f"hybrid_work_outputs_{run_stamp}"
    )
else:
    export_dir = EXPORT_ROOT

export_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Prevent recursive copying when a destination is accidentally placed
# inside a source-stage directory.
for stage_dir in SOURCE_STAGE_DIRS:
    try:
        export_dir.resolve().relative_to(
            stage_dir.resolve()
        )
        raise RuntimeError(
            "EXPORT_ROOT cannot be located inside a source-stage "
            f"directory.\nDestination: {export_dir}\n"
            f"Source: {stage_dir}"
        )
    except ValueError:
        pass


# ----------------------------------------------------------------
# 5. Select and classify files
# ----------------------------------------------------------------

def should_skip_path(path):
    return any(
        part.lower() in SKIP_DIRECTORY_NAMES
        for part in path.parts
    )


def is_map_input(path):
    return any(
        part.lower() in MAP_INPUT_DIRECTORY_NAMES
        for part in path.parts
    )


def classify_file(path):
    suffix = path.suffix.lower()
    lower_parts = {
        part.lower()
        for part in path.parts
    }

    if suffix in FIGURE_SUFFIXES:
        return "figures"

    if lower_parts & {"log", "logs"}:
        return "logs"

    if is_map_input(path):
        return "map_inputs"

    if suffix in RESULT_SUFFIXES:
        return "results"

    return "other"


def should_copy_file(path):
    if not path.is_file():
        return False

    if should_skip_path(path):
        return False

    if path.suffix.lower() not in ALLOWED_SUFFIXES:
        return False

    category = classify_file(path)

    if category == "logs" and not INCLUDE_LOGS:
        return False

    if (
        category == "map_inputs"
        and not INCLUDE_MAP_INPUTS
    ):
        return False

    return True


# ----------------------------------------------------------------
# 6. Copy files and build an inventory
# ----------------------------------------------------------------

inventory = []
copied_count = 0
skipped_existing_count = 0
failed_files = []

for source_index, stage_dir in enumerate(
    SOURCE_STAGE_DIRS,
    start=1,
):
    stage_label = (
        f"{source_index:02d}_{stage_dir.name}"
    )

    for source_file in sorted(
        stage_dir.rglob("*")
    ):
        if not should_copy_file(source_file):
            continue

        relative_path = source_file.relative_to(
            stage_dir
        )
        category = classify_file(
            source_file
        )
        destination_file = (
            export_dir
            / stage_label
            / category
            / relative_path
        )

        destination_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            destination_file.exists()
            and not OVERWRITE_EXISTING
        ):
            skipped_existing_count += 1
            continue

        try:
            shutil.copy2(
                source_file,
                destination_file,
            )

            copied_count += 1
            inventory.append({
                "source_stage": stage_dir.name,
                "category": category,
                "filename": source_file.name,
                "suffix": source_file.suffix.lower(),
                "size_bytes": source_file.stat().st_size,
                "source_path": str(Path(stage_label) / relative_path),
                "destination_path": str(destination_file.relative_to(export_dir)),
                "relative_source_path": str(relative_path),
            })

        except Exception as error:
            failed_files.append({
                "source_path": str(source_file),
                "error": repr(error),
            })


# ----------------------------------------------------------------
# 7. Write collection inventories
# ----------------------------------------------------------------

inventory_file = (
    export_dir
    / "_collection_inventory.csv"
)

inventory_fields = [
    "source_stage",
    "category",
    "filename",
    "suffix",
    "size_bytes",
    "source_path",
    "destination_path",
    "relative_source_path",
]

with inventory_file.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=inventory_fields,
    )
    writer.writeheader()
    writer.writerows(inventory)

failure_file = (
    export_dir
    / "_collection_failures.csv"
)

if failed_files:
    with failure_file.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "source_path",
                "error",
            ],
        )
        writer.writeheader()
        writer.writerows(failed_files)

category_counts = Counter(
    row["category"]
    for row in inventory
)
suffix_counts = Counter(
    row["suffix"]
    for row in inventory
)
total_size_bytes = sum(
    row["size_bytes"]
    for row in inventory
)

summary = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "analysis_root": ".",
    "source_stage_directories": [
        path.name
        for path in SOURCE_STAGE_DIRS
    ],
    "export_directory": export_dir.name,
    "copied_files": copied_count,
    "skipped_existing_files": (
        skipped_existing_count
    ),
    "failed_files": len(failed_files),
    "total_size_bytes": total_size_bytes,
    "total_size_mb": round(
        total_size_bytes / 1024 ** 2,
        2,
    ),
    "files_by_category": dict(
        sorted(category_counts.items())
    ),
    "files_by_suffix": dict(
        sorted(suffix_counts.items())
    ),
    "include_map_inputs": INCLUDE_MAP_INPUTS,
    "include_logs": INCLUDE_LOGS,
}

summary_file = (
    export_dir
    / "_collection_summary.json"
)

with summary_file.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        ensure_ascii=True,
        indent=2,
    )


# ----------------------------------------------------------------
# 8. Report completion
# ----------------------------------------------------------------

print("=" * 72)
print("Output collection completed")
print("=" * 72)
print("Destination:", export_dir.name)
print("Source output directories:")

for path in SOURCE_STAGE_DIRS:
    print("  -", path.name)

print()
print("Files copied:", copied_count)
print(
    "Existing files skipped:",
    skipped_existing_count,
)
print("Copy failures:", len(failed_files))
print(
    "Collected file size:",
    f"{total_size_bytes / 1024 ** 2:,.2f} MB",
)

print("\nFiles by category:")
for category, count in sorted(
    category_counts.items()
):
    print(
        f"  {category}: {count}"
    )

print("\nInventory:", inventory_file)
print("Summary:", summary_file)

if failed_files:
    print("Failure log:", failure_file)

print("=" * 72)
